# Scheduler Optimization Guide: One Layer at a Time

This notebook is a focused learning companion for the twenty optimization layers built on top of
the frozen FIFO singleton scheduler for
[Codeforces 2251A](https://codeforces.com/contest/2251/problem/A).

It is deliberately separate from:

- `edge_cloud_scheduling_lab.ipynb`, which teaches the whole problem and protocol; and
- `scheduler_benchmark_workbench.ipynb`, which compares every version across the full suite.

Here the question is narrower: **what problem does each optimization solve, why should it
work, what can go wrong, and what did it change in a controlled local example?**

## Goal

For every layer, we will connect five things:

1. the limitation in the previous scheduler;
2. the scheduling intuition;
3. the actual implemented decision rule;
4. correctness invariants and tradeoffs; and
5. an executable comparison of `v(N-1)` versus `vN` on one isolation scenario.

Versions 1–18 are cumulative: `v4` means layers 1, 2, 3, **and** 4 are enabled. Layer 19
deliberately branches from the promoted v15 policy so its terminal-stage experiment does not
inherit the rejected learned-grouping behavior in layers 16–18. Layer 20 extends that terminal
branch backward through D PROC. Each comparison uses the predecessor recorded in the layer map.

## Setup

The notebook reads the checked-in registry, C++ sources, task tables, scenarios, and local
judge. It compiles the frozen versions itself, so the displayed measurements are not copied
from an older report.

In [1]:
from __future__ import annotations

import html
import json
import math
import os
import shlex
import subprocess
import sys
from pathlib import Path
from typing import Any, Iterable

from IPython.display import Code, HTML, Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "main.cpp").is_file() and (candidate / "tools/local_judge.py").is_file():
            return candidate
    raise FileNotFoundError("Could not find the scheduler repository")


REPO_ROOT = find_repo_root()
REGISTRY_PATH = REPO_ROOT / "scheduler_versions/registry.json"
SCENARIO_DIR = REPO_ROOT / "scenarios"
LAYERED_SOURCE_PATH = REPO_ROOT / "scheduler_versions/layered_scheduler.cpp"
BASELINE_SOURCE_PATH = REPO_ROOT / "scheduler_versions/v0_baseline.cpp"
TUNING_REPORT_PATH = REPO_ROOT / "benchmarks/learned-grouping-policy.json"
FURTHER_REPORT_PATH = REPO_ROOT / "benchmarks/further-optimization-experiments.json"
BUILD_DIR = REPO_ROOT / "build/optimization-guide"
RESULT_DIR = BUILD_DIR / "results"

BUILD_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository: {REPO_ROOT}")
print(f"Build area: {BUILD_DIR.relative_to(REPO_ROOT)}")

Repository: /Users/aadikhanna/Github/AI Optimizer/CodeForces-Edge-Collaborative-Scheduling
Build area: build/optimization-guide


In [2]:
def display_table(
    rows: Iterable[dict[str, Any]], columns: list[tuple[str, str]] | None = None
) -> None:
    bounded_rows = list(rows)
    if not bounded_rows:
        display(Markdown("_No rows._"))
        return
    if columns is None:
        columns = [(key, key) for key in bounded_rows[0]]
    header = "".join(f"<th>{html.escape(label)}</th>" for _, label in columns)
    body = []
    for row in bounded_rows:
        cells = "".join(
            f"<td>{html.escape(str(row.get(key, '')))}</td>" for key, _ in columns
        )
        body.append(f"<tr>{cells}</tr>")
    display(
        HTML(
            "<table><thead><tr>"
            + header
            + "</tr></thead><tbody>"
            + "".join(body)
            + "</tbody></table>"
        )
    )


def run_command(command: list[str], timeout_seconds: float = 180.0) -> subprocess.CompletedProcess[str]:
    completed = subprocess.run(
        command,
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
        timeout=timeout_seconds,
    )
    if completed.returncode != 0:
        detail = completed.stderr.strip() or completed.stdout.strip()
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(command)}\n{detail}")
    return completed


def source_between(source: str, start_marker: str, end_marker: str, max_lines: int = 100) -> str:
    start = source.index(start_marker)
    end = source.index(end_marker, start)
    lines = source[start:end].rstrip().splitlines()
    if len(lines) > max_lines:
        lines = lines[:max_lines] + ["// ... bounded notebook preview ..."]
    return "\n".join(lines)


def source_window(source: str, marker: str, lines_after: int = 45) -> str:
    lines = source.splitlines()
    start = next(index for index, line in enumerate(lines) if marker in line)
    return "\n".join(lines[start : start + lines_after])


def display_source(snippet: str) -> None:
    display(Code(snippet, language="cpp"))

## The twenty-layer map

The target scenario for each layer is intentionally constructed to make one pressure visible.
It is a mechanism test, not a prediction of the official hidden-test distribution.

In [3]:
LAYER_SPECS = [
    {
        "layer": 1,
        "previous": "v0-baseline",
        "current": "v1-multi-active",
        "title": "Multiple active requests per cloud",
        "target": "two_cloud_parallel",
        "decision": "Cloud utilization",
    },
    {
        "layer": 2,
        "previous": "v1-multi-active",
        "current": "v2-load-aware",
        "title": "Observable-load-aware placement",
        "target": "output_length_skew",
        "decision": "Cloud selection",
    },
    {
        "layer": 3,
        "previous": "v2-load-aware",
        "current": "v3-immediate-groups",
        "title": "Immediate decode grouping",
        "target": "batch_friendly_burst",
        "decision": "Group formation",
    },
    {
        "layer": 4,
        "previous": "v3-immediate-groups",
        "current": "v4-table-groups",
        "title": "Task-table-aware group size",
        "target": "nonmonotonic_batch_table",
        "decision": "Group size",
    },
    {
        "layer": 5,
        "previous": "v4-table-groups",
        "current": "v5-slo-aware",
        "title": "SLO urgency and bounded waiting",
        "target": "slo_priority_collision",
        "decision": "Priority and pacing",
    },
    {
        "layer": 6,
        "previous": "v5-slo-aware",
        "current": "v6-prefill-chunks",
        "title": "Adaptive prefill chunks",
        "target": "single_cloud_prefill_interleave",
        "decision": "Task granularity",
    },
    {
        "layer": 7,
        "previous": "v6-prefill-chunks",
        "current": "v7-link-aware",
        "title": "Score- and link-aware scheduling",
        "target": "latency_weighted_slow_link",
        "decision": "Shared-link pressure",
    },
    {
        "layer": 8,
        "previous": "v7-link-aware",
        "current": "v8-exact-timelines",
        "title": "Exact virtual timelines",
        "target": "exact_wait_horizon",
        "decision": "Event-bounded waiting",
    },
    {
        "layer": 9,
        "previous": "v8-exact-timelines",
        "current": "v9-fanout-cohorts",
        "title": "Fanout- and cohort-aware grouping",
        "target": "cross_cloud_fanout",
        "decision": "D PRE membership",
    },
    {
        "layer": 10,
        "previous": "v9-fanout-cohorts",
        "current": "v10-batch-placement",
        "title": "Batch-aware cloud placement",
        "target": "batch_aware_placement",
        "decision": "Pack versus spread",
    },
    {
        "layer": 11,
        "previous": "v10-batch-placement",
        "current": "v11-score-slack",
        "title": "Predicted score slack",
        "target": "predicted_deadline_slack",
        "decision": "Milestone priority",
    },
    {
        "layer": 12,
        "previous": "v11-score-slack",
        "current": "v12-deadline-chunks",
        "title": "Deadline-aware chunks",
        "target": "chunk_deadline_collision",
        "decision": "Prefill piece duration",
    },
    {
        "layer": 13,
        "previous": "v12-deadline-chunks",
        "current": "v13-attained-service",
        "title": "Attained-service scheduling",
        "target": "attained_service_tail",
        "decision": "Decode member selection",
    },
    {
        "layer": 14,
        "previous": "v13-attained-service",
        "current": "v14-backpressure",
        "title": "Link backpressure",
        "target": "downstream_backpressure",
        "decision": "Drain versus inject",
    },
    {
        "layer": 15,
        "previous": "v14-backpressure",
        "current": "v15-one-token-lookahead",
        "title": "Bounded one-token lookahead",
        "target": "one_token_lookahead",
        "decision": "End-to-end group cost",
    },
    {
        "layer": 16,
        "previous": "v15-one-token-lookahead",
        "current": "v16-counterfactual-groups",
        "title": "Counterfactual decode grouping",
        "target": "counterfactual_grouping",
        "decision": "Candidate group value",
    },
    {
        "layer": 17,
        "previous": "v16-counterfactual-groups",
        "current": "v17-learned-group-ranker",
        "title": "Offline-fitted group ranker",
        "target": "learned_grouping_recovery",
        "decision": "Group-value coefficients",
    },
    {
        "layer": 18,
        "previous": "v17-learned-group-ranker",
        "current": "v18-nonlinear-group-ranker",
        "title": "Nonlinear interaction audit",
        "target": "nonlinear_ranker_holdout",
        "decision": "Model complexity",
    },
    {
        "layer": 19,
        "previous": "v15-one-token-lookahead",
        "current": "v19-terminal-dpost",
        "title": "Remainder-aware terminal D POST",
        "target": "terminal_dpost_remainder",
        "decision": "Finite-queue clearance",
    },
    {
        "layer": 20,
        "previous": "v19-terminal-dpost",
        "current": "v20-terminal-dproc",
        "title": "Stage-correct terminal D PROC",
        "target": "terminal_dproc_clearance",
        "decision": "D PROC-to-D POST clearance",
    },
]

display_table(
    LAYER_SPECS,
    [
        ("layer", "Layer"),
        ("title", "Optimization"),
        ("decision", "Decision changed"),
        ("target", "Isolation scenario"),
    ],
)

Layer,Optimization,Decision changed,Isolation scenario
1,Multiple active requests per cloud,Cloud utilization,two_cloud_parallel
2,Observable-load-aware placement,Cloud selection,output_length_skew
3,Immediate decode grouping,Group formation,batch_friendly_burst
4,Task-table-aware group size,Group size,nonmonotonic_batch_table
5,SLO urgency and bounded waiting,Priority and pacing,slo_priority_collision
6,Adaptive prefill chunks,Task granularity,single_cloud_prefill_interleave
7,Score- and link-aware scheduling,Shared-link pressure,latency_weighted_slow_link
8,Exact virtual timelines,Event-bounded waiting,exact_wait_horizon
9,Fanout- and cohort-aware grouping,D PRE membership,cross_cloud_fanout
10,Batch-aware cloud placement,Pack versus spread,batch_aware_placement


## Build adjacent versions and generate fresh evidence

Every frozen version is compiled with identical base flags. Layers 1–20 use the same source
file with `OPT_LEVEL=N`; v0 remains a separate frozen implementation. The source gates make
level 19 start from v15 rather than enabling levels 16–18.

In [4]:
registry = json.loads(REGISTRY_PATH.read_text())
registered_versions = {version["name"]: version for version in registry["versions"]}
scenario_paths = sorted(SCENARIO_DIR.glob("*.json"))
scenario_path_by_name = {
    json.loads(path.read_text())["name"]: path for path in scenario_paths
}
scenario_data = {
    name: json.loads(path.read_text()) for name, path in scenario_path_by_name.items()
}

required_version_names = ["v0-baseline"] + [spec["current"] for spec in LAYER_SPECS]
CXX = os.environ.get("CXX", "g++")
CXXFLAGS = shlex.split(
    os.environ.get("CXXFLAGS", "-std=c++17 -O2 -pipe -Wall -Wextra -Wpedantic")
)

executables: dict[str, Path] = {}
build_rows = []
for version_name in required_version_names:
    version = registered_versions[version_name]
    source_path = REPO_ROOT / version["source"]
    executable = BUILD_DIR / version_name
    define_flags = [f"-D{define}" for define in version.get("compile_defines", [])]
    completed = run_command(
        [CXX, *CXXFLAGS, *define_flags, str(source_path), "-o", str(executable)]
    )
    executables[version_name] = executable
    build_rows.append(
        {
            "version": version_name,
            "layer": version.get("layer", 0),
            "gate": ", ".join(version.get("compile_defines", [])) or "standalone v0",
            "warnings": sum("warning:" in line for line in completed.stderr.splitlines()),
            "status": "PASS",
        }
    )

display_table(build_rows)

version,layer,gate,warnings,status
v0-baseline,0,standalone v0,0,PASS
v1-multi-active,1,OPT_LEVEL=1,0,PASS
v2-load-aware,2,OPT_LEVEL=2,0,PASS
v3-immediate-groups,3,OPT_LEVEL=3,0,PASS
v4-table-groups,4,OPT_LEVEL=4,0,PASS
v5-slo-aware,5,OPT_LEVEL=5,0,PASS
v6-prefill-chunks,6,OPT_LEVEL=6,0,PASS
v7-link-aware,7,OPT_LEVEL=7,0,PASS
v8-exact-timelines,8,OPT_LEVEL=8,0,PASS
v9-fanout-cohorts,9,OPT_LEVEL=9,0,PASS


In [5]:
def clamp01(value: float) -> float:
    return max(0.0, min(1.0, value))


def recompute_score(result: dict[str, Any], scoring: dict[str, Any]) -> float:
    excess_tdr = max(0.0, (result["tdr"] - scoring["SLO1"]) / scoring["SLO1"])
    excess_tpot = max(0.0, (result["tpot"] - scoring["SLO2"]) / scoring["SLO2"])
    distance = math.hypot(excess_tdr, excess_tpot)
    throughput_component = clamp01(
        (result["throughput"] - scoring["tp_base"])
        / (scoring["tp_UB"] - scoring["tp_base"])
    )
    distance_base = scoring["dist_base"]
    latency_component = (
        max(0.0, 1.0 - distance / distance_base)
        if distance_base > 0
        else (1.0 if distance == 0 else 0.0)
    )
    return 1000.0 * (
        scoring["w_tp"] * throughput_component + scoring["w_c"] * latency_component
    )


target_results: dict[tuple[str, str], dict[str, Any]] = {}
requested_runs = {
    (version_name, spec["target"])
    for spec in LAYER_SPECS
    for version_name in (spec["previous"], spec["current"])
}

maximum_score_error = 0.0
for version_name, scenario_name in sorted(requested_runs):
    result_path = RESULT_DIR / f"{version_name}--{scenario_name}.json"
    run_command(
        [
            "python3",
            "tools/local_judge.py",
            "--solver",
            str(executables[version_name]),
            "--scenarios",
            str(scenario_path_by_name[scenario_name]),
            "--json-out",
            str(result_path),
        ]
    )
    rows = json.loads(result_path.read_text())
    assert len(rows) == 1
    result = rows[0]
    scenario = scenario_data[scenario_name]
    expected_tokens = sum(request["output_length"] for request in scenario["requests"])
    assert result["legal"] and result["tokens"] == expected_tokens
    score_error = abs(recompute_score(result, scenario["scoring"]) - result["score"])
    maximum_score_error = max(maximum_score_error, score_error)
    target_results[(version_name, scenario_name)] = result

assert maximum_score_error < 1e-7
print(
    f"Generated {len(target_results)} legal adjacent-version runs; "
    f"maximum independent score error={maximum_score_error:.2e}"
)

Generated 40 legal adjacent-version runs; maximum independent score error=0.00e+00


In [6]:
def percent_change(after: float, before: float) -> float | None:
    return 100.0 * (after / before - 1.0) if before != 0 else None


evidence_by_layer: dict[int, dict[str, Any]] = {}
for spec in LAYER_SPECS:
    before = target_results[(spec["previous"], spec["target"])]
    after = target_results[(spec["current"], spec["target"])]
    evidence_by_layer[spec["layer"]] = {
        "layer": spec["layer"],
        "scenario": spec["target"],
        "before version": spec["previous"],
        "after version": spec["current"],
        "score before": before["score"],
        "score after": after["score"],
        "score delta": after["score"] - before["score"],
        "throughput delta %": percent_change(after["throughput"], before["throughput"]),
        "TDR delta %": percent_change(after["tdr"], before["tdr"]),
        "TPOT delta %": percent_change(after["tpot"], before["tpot"]),
        "elapsed delta %": percent_change(after["elapsed"], before["elapsed"]),
    }


def format_percent(value: float | None) -> str:
    return "n/a" if value is None else f"{value:+.1f}%"


def display_layer_evidence(layer: int) -> None:
    raw = evidence_by_layer[layer]
    scenario = scenario_data[raw["scenario"]]
    display(Markdown(f"**Isolation case:** `{raw['scenario']}` — {scenario['description']}"))
    display_table(
        [
            {
                "comparison": f"{raw['before version']} → {raw['after version']}",
                "score": f"{raw['score before']:.3f} → {raw['score after']:.3f}",
                "score delta": f"{raw['score delta']:+.3f}",
                "throughput Δ": format_percent(raw["throughput delta %"]),
                "TDR Δ": format_percent(raw["TDR delta %"]),
                "TPOT Δ": format_percent(raw["TPOT delta %"]),
                "elapsed Δ": format_percent(raw["elapsed delta %"]),
            }
        ]
    )
    display(
        Markdown(
            "_Score and throughput: higher is better. TDR, TPOT, and elapsed time: "
            "lower is better. This adjacent comparison supports only the constructed case._"
        )
    )


layered_source = LAYERED_SOURCE_PATH.read_text()
baseline_source = BASELINE_SOURCE_PATH.read_text()

## Before optimizing: separate assignment from execution

This distinction drives the whole design:

- **Assigned to cloud C:** the request's `P PROC` and future `D PROC` work must use C.
- **Executing on cloud C:** C is currently occupied by one scheduled task.
- **Queued for cloud C:** the request belongs to C but is waiting in a legal state.

A cloud may own many unfinished requests, but can execute only one task at a time. Likewise,
all clouds share one FIFO `UP` link and one FIFO `DOWN` link, so adding cloud compute
parallelism does not add transfer-link parallelism.

The scheduler also cannot see output lengths. `FIN` reveals that a request has ended, but
before then we must estimate future decode load from observable state rather than predict the
exact number of remaining tokens.

## Layer 0 — the reference we are improving

### Previous policy

The frozen baseline admits requests FIFO, reserves one whole cloud per unfinished request,
runs one full prefill piece, and makes every decode group a singleton.

### Why start this simply?

It minimizes bookkeeping and makes illegal transitions easier to detect. Its weakness is
intentional: a cloud reservation remains occupied conceptually even while its request is on
edge `E` or a transfer link, so useful cloud compute can sit idle.

### Reference pseudocode

```text
when edge is free and a cloud reservation is free:
    take oldest arrived request
    assign it to first free reservation
    run singleton lifecycle until FIN
    only then reuse that cloud reservation
```

In [7]:
display_source(
    source_between(
        baseline_source,
        "string dispatch_admission()",
        "string dispatch_cloud_task",
        max_lines=85,
    )
)

string dispatch_admission() {
        int request_id = pending_requests_.front();
        pending_requests_.pop_front();

        int cloud = free_clouds_.front();
        free_clouds_.pop_front();

        Request& req = request(request_id);
        expect_state(req, RequestState::WAITING_FOR_CLOUD, "P PRE dispatch");
        if (cloud_reserved_[cloud]) {
            fail("admission selected a reserved cloud");
        }

        req.cloud = cloud;
        req.state = RequestState::P_PRE_RUNNING;
        cloud_reserved_[cloud] = true;
        edge_busy_ = true;

        return "E P PRE " + to_string(cloud) + " " + to_string(request_id);
    }

    string dispatch_edge_task() {
        ReadyTask task = edge_ready_.front();
        edge_ready_.pop_front();
        Request& req = request(task.request_id);

        edge_busy_ = true;
        switch (task.kind) {
            case TaskKind::P_POST:
                expect_state(req, RequestState::P_POST_READY, "P POST dispatch");
                req.state = RequestState::P_POST_RUNNING;
                return "E P POST " + to_string(req.cloud) + " " + to_string(req.id);
            case TaskKind::D_PRE:
                expect_state(req, RequestState::D_PRE_READY, "D PRE dispatch");
                req.state = RequestState::D_PRE_RUNNING;
                return "E D PRE -1 1 " + to_string(req.id);
            case TaskKind::D_POST:
                expect_state(req, RequestState::D_POST_READY, "D POST dispatch");
                req.state = RequestState::D_POST_RUNNING;
                return "E D POST -1 1 " + to_string(req.id);
            case TaskKind::P_PROC:
            case TaskKind::D_PROC:
                fail("cloud task appeared in the edge queue");
        }
        fail("unreachable edge task kind");
    }

## Layer 1 — multiple active requests per cloud

### Bottleneck

The baseline confuses “one executing task” with “one assigned request.” While request A is
doing edge or link work, its cloud can legally process request B—but the reservation policy
prevents that.

### Intuition

Turn each cloud into a small pipeline. Multiple requests may be at different lifecycle stages:

```text
request A: waiting for DOWN transfer
request B: D PROC ready       ← cloud can run this
request C: P PROC queued
```

A single `cloud_busy[cloud]` flag still enforces compute capacity. Per-cloud ready queues hold
legal `P PROC` and `D PROC` work. At this layer, admission uses round-robin so the change tests
utilization without yet adding a load model.

### Why it can help

More assigned work raises the probability that a free cloud has something ready. That reduces
idle gaps, improves throughput, and often reduces total completion time.

### Correctness invariants

- A request keeps its assigned cloud for its entire lifecycle.
- Only one task executes on a cloud at once.
- A `D PROC` group contains members from only that cloud.

### Tradeoff

Round-robin balances request counts, not remaining work. Hidden output lengths can still create
severe skew, and deeper queues may worsen individual token gaps.

In [8]:
display_source(
    source_between(layered_source, "double cloud_load_score", "double observed_request_urgency", 85)
)
display_layer_evidence(1)

double cloud_load_score(int cloud) const {
        const double remaining_busy = max(0.0, cloud_busy_until_[cloud] - current_time_);
        const double decode_proxy =
            active_requests_[cloud] * (schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1));
        const int ready_decode = static_cast<int>(d_proc_ready_[cloud].size());
        const double ready_decode_work =
            ready_decode > 0
                ? schedule_cost_ + duration(DurationColumn::DECODE_PROC, ready_decode)
                : 0.0;
        return remaining_busy + pending_prefill_work_[cloud] + ready_decode_work +
               0.35 * decode_proxy;
    }

    double batch_aware_cloud_score(const Request& req, int cloud) const {
        (void)req;
        const int prospective_cohort = max(1, active_requests_[cloud] + 1);
        const int candidate_size = min(
            prospective_cohort,
            best_group_size(DurationColumn::DECODE_PROC, prospective_cohort)
        );
        const double singleton = schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1);
        const double grouped_per_request =
            (schedule_cost_ + duration(DurationColumn::DECODE_PROC, candidate_size)) /
            candidate_size;
        const double savings_per_iteration = max(0.0, singleton - grouped_per_request);
        const double efficiency_ratio = singleton / max(1e-9, grouped_per_request);
        const double cohort_strength = min(
            4,
            active_decode_requests_[cloud] + static_cast<int>(d_proc_ready_[cloud].size())
        );
        const int minimum_active = *min_element(active_requests_.begin(), active_requests_.end());
        const bool seed_decode_cohort = minimum_active == 0 &&
                                        active_requests_[cloud] == 1 &&
                                        active_decode_requests_[cloud] > 0;
        const double credit_scale = seed_decode_cohort ? 1.0 : 0.2;
        const double credit_cap = seed_decode_cohort ? 2.0 * singleton : 0.5 * singleton;
        const double batch_credit =
            efficiency_ratio >= 3.0 &&
                    active_requests_[cloud] <= minimum_active + 1
            ? min(
                  credit_cap,
                  credit_scale * throughput_weight_ * (1.0 + 0.5 * cohort_strength) *
                      savings_per_iteration
              )
            : 0.0;
        return cloud_load_score(cloud) - batch_credit;
    }

    int choose_cloud(const Request& req) {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel == 1) {
            const int cloud = next_round_robin_cloud_;
            next_round_robin_cloud_ = (next_round_robin_cloud_ + 1) % cloud_count_;
            return cloud;
        }
        // SUBMISSION_FEATURE_END pre20_only

        int best_cloud = 0;
        double best_load = kOptimizationLevel >= 10
            ? batch_aware_cloud_score(req, 0)
            : cloud_load_score(0);
        for (int cloud = 1; cloud < cloud_count_; ++cloud) {
            const double load = kOptimizationLevel >= 10
                ? batch_aware_cloud_score(req, cloud)
                : cloud_load_score(cloud);
            if (load + 1e-12 < best_load) {
                best_load = load;
                best_cloud = cloud;
            }
        }
        return best_cloud;
    }

**Isolation case:** `two_cloud_parallel` — A six-request burst that exposes cloud reservation and edge serialization.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v0-baseline → v1-multi-active,776.939 → 1000.000,+223.061,+78.2%,-65.0%,+43.2%,-43.9%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 2 — observable-load-aware cloud placement

### Bottleneck

Two clouds with the same request count can have very different visible work. One may be busy,
have a long prefill queued, and own several decode streams; the other may be nearly empty.
Round-robin cannot see that difference.

### Implemented estimate

Before dispatching `P PRE`, score every cloud:

$$
\text{load}(c)=\text{remaining busy time}+\text{known prefill work}
+\text{ready decode work}+0.35\times\text{active-request proxy}.
$$

The first three terms measure work already visible. The last term prevents requests currently
on the edge/links from disappearing from the estimate. Its coefficient is intentionally below
1 because the number of future output tokens remains hidden.

### Toy example

The numbers below are illustrative components in milliseconds, not a replayed judge frame.

In [9]:
display_source(
    source_between(layered_source, "double cloud_load_score", "double request_urgency", 70)
)

toy_clouds = [
    {"cloud": "C0", "busy": 40.0, "prefill": 80.0, "ready decode": 8.0, "active proxy": 12.0},
    {"cloud": "C1", "busy": 5.0, "prefill": 20.0, "ready decode": 0.0, "active proxy": 16.0},
]
for row in toy_clouds:
    row["estimated load"] = (
        row["busy"] + row["prefill"] + row["ready decode"] + 0.35 * row["active proxy"]
    )
    row["chosen"] = "yes" if row["cloud"] == "C1" else ""
display_table(toy_clouds)

double cloud_load_score(int cloud) const {
        const double remaining_busy = max(0.0, cloud_busy_until_[cloud] - current_time_);
        const double decode_proxy =
            active_requests_[cloud] * (schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1));
        const int ready_decode = static_cast<int>(d_proc_ready_[cloud].size());
        const double ready_decode_work =
            ready_decode > 0
                ? schedule_cost_ + duration(DurationColumn::DECODE_PROC, ready_decode)
                : 0.0;
        return remaining_busy + pending_prefill_work_[cloud] + ready_decode_work +
               0.35 * decode_proxy;
    }

    double batch_aware_cloud_score(const Request& req, int cloud) const {
        (void)req;
        const int prospective_cohort = max(1, active_requests_[cloud] + 1);
        const int candidate_size = min(
            prospective_cohort,
            best_group_size(DurationColumn::DECODE_PROC, prospective_cohort)
        );
        const double singleton = schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1);
        const double grouped_per_request =
            (schedule_cost_ + duration(DurationColumn::DECODE_PROC, candidate_size)) /
            candidate_size;
        const double savings_per_iteration = max(0.0, singleton - grouped_per_request);
        const double efficiency_ratio = singleton / max(1e-9, grouped_per_request);
        const double cohort_strength = min(
            4,
            active_decode_requests_[cloud] + static_cast<int>(d_proc_ready_[cloud].size())
        );
        const int minimum_active = *min_element(active_requests_.begin(), active_requests_.end());
        const bool seed_decode_cohort = minimum_active == 0 &&
                                        active_requests_[cloud] == 1 &&
                                        active_decode_requests_[cloud] > 0;
        const double credit_scale = seed_decode_cohort ? 1.0 : 0.2;
        const double credit_cap = seed_decode_cohort ? 2.0 * singleton : 0.5 * singleton;
        const double batch_credit =
            efficiency_ratio >= 3.0 &&
                    active_requests_[cloud] <= minimum_active + 1
            ? min(
                  credit_cap,
                  credit_scale * throughput_weight_ * (1.0 + 0.5 * cohort_strength) *
                      savings_per_iteration
              )
            : 0.0;
        return cloud_load_score(cloud) - batch_credit;
    }

    int choose_cloud(const Request& req) {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel == 1) {
            const int cloud = next_round_robin_cloud_;
            next_round_robin_cloud_ = (next_round_robin_cloud_ + 1) % cloud_count_;
            return cloud;
        }
        // SUBMISSION_FEATURE_END pre20_only

        int best_cloud = 0;
        double best_load = kOptimizationLevel >= 10
            ? batch_aware_cloud_score(req, 0)
            : cloud_load_score(0);
        for (int cloud = 1; cloud < cloud_count_; ++cloud) {
            const double load = kOptimizationLevel >= 10
                ? batch_aware_cloud_score(req, cloud)
                : cloud_load_score(cloud);
            if (load + 1e-12 < best_load) {
                best_load = load;
                best_cloud = cloud;
            }
        }
// ... bounded notebook preview ...

cloud,busy,prefill,ready decode,active proxy,estimated load,chosen
C0,40.0,80.0,8.0,12.0,132.2,
C1,5.0,20.0,0.0,16.0,30.6,yes


### Why it can help

Visible heavy work is routed away from the cloud least able to start it soon. The proxy also
avoids choosing a cloud that only *looks* empty because its requests are temporarily elsewhere.

### Tradeoff

This is not true remaining processing time. A one-token request and a thousand-token request
initially have the same hidden decode future. Placement is permanent, so an early estimation
error cannot be repaired by migrating the request later.

In [10]:
display_layer_evidence(2)

**Isolation case:** `output_length_skew` — Two early requests have very different hidden output lengths; later requests queue behind their cloud reservations.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v1-multi-active → v2-load-aware,695.922 → 696.830,+0.907,+0.3%,+0.0%,-1.2%,-0.3%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 3 — immediately group compatible decode work

### Bottleneck

Each assignment pays fixed scheduling cost `S`. Running eight singleton tasks pays that cost
eight times. A group pays it once and may also use a sublinear task duration.

### Implemented rule

When the relevant resource becomes free, take every compatible request ready **now**:

- edge `D PRE`: may group across clouds;
- cloud `D PROC`: members must share that cloud;
- edge `D POST`: may group across clouds.

Group membership lasts for one stage of one decode iteration. It is not a permanent batch.
This layer does not wait for future arrivals; that is a separate layer-5 decision.

In [11]:
display_source(source_window(layered_source, "if (candidate.kind == TaskKind::D_PRE)", 36))

if (candidate.kind == TaskKind::D_PRE) {
            return best_group_size(
                DurationColumn::DECODE_PRE, static_cast<int>(d_pre_ready_.size())
            );
        }
        if (candidate.kind == TaskKind::D_POST) {
            return best_group_size(
                DurationColumn::DECODE_POST, static_cast<int>(d_post_ready_.size())
            );
        }
        if (candidate.kind == TaskKind::D_PROC) {
            const int cloud = request(candidate.request_id).cloud;
            return best_group_size(
                DurationColumn::DECODE_PROC,
                static_cast<int>(d_proc_ready_[cloud].size())
            );
        }
        return 1;
    }

    double rollout_candidate_value(const Candidate& candidate, double start_delay) const {
        const Request& req = request(candidate.request_id);
        const int group_size = rollout_group_size(candidate);
        const double service = action_service_time(candidate.kind, group_size, req);
        const bool prefill = candidate.kind == TaskKind::P_PRE ||
                             candidate.kind == TaskKind::P_PROC ||
                             candidate.kind == TaskKind::P_POST;
        const double milestone = prefill
            ? estimated_prefill_path(candidate.kind, req)
            : estimated_decode_path(candidate.kind, group_size);
        const double observed = prefill
            ? current_time_ - req.arrival_time
            : current_time_ - req.decode_clock_start;
        const double slo = prefill ? slo_tdr_ : slo_tpot_;
        const double ratio = (observed + start_delay + milestone) / max(1e-9, slo);
        const double excess = max(0.0, ratio - 1.0);

### Why grouping amortizes overhead

In the batch-friendly scenario, `S=8 ms`, singleton `D PROC=3 ms`, and size-8
`D PROC=11 ms`. Compare eight singleton services with one group:

In [12]:
batch_scenario = scenario_data["batch_friendly_burst"]
batch_rows_path = SCENARIO_DIR / batch_scenario["task_times_file"]
batch_rows = json.loads(batch_rows_path.read_text())["task_times"]
decode_proc_by_size = {row["batch_size"]: row["decode_proc"] for row in batch_rows}
schedule_cost = batch_scenario["system"]["S"]
singleton_cost = 8 * (schedule_cost + decode_proc_by_size[1])
group_cost = schedule_cost + decode_proc_by_size[8]
display_table(
    [
        {"plan": "8 singleton D PROC tasks", "service ms": singleton_cost, "members/ms": f"{8/singleton_cost:.3f}"},
        {"plan": "1 size-8 D PROC group", "service ms": group_cost, "members/ms": f"{8/group_cost:.3f}"},
    ]
)
print(f"Idealized D PROC service-rate gain: {singleton_cost / group_cost:.2f}x")

plan,service ms,members/ms
8 singleton D PROC tasks,88.0,0.091
1 size-8 D PROC group,19.0,0.421


Idealized D PROC service-rate gain: 4.63x


### Tradeoff

The largest ready group can occupy a resource longer, convoy urgent requests, or land on a
poor region of the task-time table. Immediate grouping removes repeated overhead, but it does
not yet answer which group size is best.

In [13]:
display_layer_evidence(3)

**Isolation case:** `batch_friendly_burst` — A high-overhead request burst where decode grouping should materially improve throughput.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v2-load-aware → v3-immediate-groups,178.692 → 676.030,+497.338,+332.7%,+0.0%,-86.6%,-76.9%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 4 — choose group size from the task-time table

### Bottleneck

“Largest group” is only good when the duration curve scales well. The judge can provide a
nonmonotonic curve where size 8 is slower per member than size 4.

### Implemented objective

For each decode stage, choose the available size `b` maximizing local service rate:

$$
\text{rate}(b)=\frac{b}{S+T_{\text{stage}}(b)}.
$$

Candidate sizes include 1, all ready members, and task-table breakpoints plus neighboring
integers. `-1` values are ignored per column and usable points are linearly interpolated.
Smaller groups win exact rate ties.

In [14]:
display_source(
    source_between(layered_source, "int best_group_size", "bool should_wait_for_group", 90)
)

int best_group_size(DurationColumn column, int available) const {
        if (available <= 1 || kOptimizationLevel < 3) {
            return 1;
        }
        // SUBMISSION_FEATURE_BEGIN experimental_grouping
        if constexpr (kExperimentalGrouping) {
            const vector<int>& cache = best_group_size_cache_[static_cast<int>(column)];
            return cache[min<int>(available, cache.size() - 1)];
        }
        // SUBMISSION_FEATURE_END experimental_grouping
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel == 3) {
            return available;
        }
        // SUBMISSION_FEATURE_END pre20_only

        set<int> candidates = {1, available};
        for (const auto& [size, ignored] : duration_curves_[static_cast<int>(column)]) {
            (void)ignored;
            for (int candidate : {size - 1, size, size + 1}) {
                if (1 <= candidate && candidate <= available) {
                    candidates.insert(candidate);
                }
            }
        }

        int best_size = 1;
        double best_rate = -1;
        for (int size : candidates) {
            double service_time = schedule_cost_ + duration(column, size);
            // SUBMISSION_FEATURE_BEGIN pre20_only
            if constexpr (kOptimizationLevel >= 7) {
            // SUBMISSION_FEATURE_END pre20_only
                if (column == DurationColumn::DECODE_PRE ||
                    column == DurationColumn::DECODE_PROC) {
                    service_time += transfer_time(
                        static_cast<long long>(size) * bytes_per_token_
                    );
                }
            // SUBMISSION_FEATURE_BEGIN pre20_only
            }
            // SUBMISSION_FEATURE_END pre20_only
            // SUBMISSION_FEATURE_BEGIN pre20_only
            if constexpr (kOptimizationLevel >= 15) {
            // SUBMISSION_FEATURE_END pre20_only
                if (column == DurationColumn::DECODE_PRE &&
                    downstream_group_is_hostile(size)) {
                    service_time += schedule_cost_ +
                                    duration(DurationColumn::DECODE_PROC, size) +
                                    transfer_time(
                                        static_cast<long long>(size) * bytes_per_token_
                                    ) +
                                    schedule_cost_ +
                                    duration(DurationColumn::DECODE_POST, size);
                } else if (column == DurationColumn::DECODE_PROC &&
                           downstream_group_is_hostile(size)) {
                    service_time += schedule_cost_ +
                                    duration(DurationColumn::DECODE_POST, size);
                }
            // SUBMISSION_FEATURE_BEGIN pre20_only
            }
            // SUBMISSION_FEATURE_END pre20_only
            const double rate = size / service_time;
            if (rate > best_rate + 1e-12 ||
                (abs(rate - best_rate) <= 1e-12 && size < best_size)) {
                best_rate = rate;
                best_size = size;
            }
        }
        return best_size;
    }

    // SUBMISSION_FEATURE_BEGIN experimental_grouping
    vector<int> bounded_candidate_group_sizes(
        DurationColumn column,
        int available
    ) const {
        set<int> sizes = {1, available};
        const int best = best_group_size(column, available);
        for (int candidate : {
                 best - 1,
                 best,
                 best + 1,
                 best / 2,
                 min(available, 2 * best),
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (1 <= candidate && candidate <= available) {
// ... bounded notebook preview ...

### See the nonmonotonic choice

In [15]:
nonmonotonic = scenario_data["nonmonotonic_batch_table"]
nonmonotonic_rows = nonmonotonic["task_times"]
nonmonotonic_proc = {row["batch_size"]: row["decode_proc"] for row in nonmonotonic_rows}
nonmonotonic_cost = nonmonotonic["system"]["S"]
rate_rows = []
for size in (1, 4, 8):
    service = nonmonotonic_cost + nonmonotonic_proc[size]
    rate_rows.append(
        {
            "group size": size,
            "S + D PROC ms": f"{service:.1f}",
            "members/ms": f"{size/service:.3f}",
            "selected among ≤8": "yes" if size == 4 else "",
        }
    )
display_table(rate_rows)

group size,S + D PROC ms,members/ms,selected among ≤8
1,4.0,0.250,
4,5.0,0.800,yes
8,21.0,0.381,


### Tradeoff

This is a local stage objective. It cannot see future arrivals or fully model downstream edge
queues and collective links. It may also leave a small remainder group. Layer 7 later adds a
transfer-time term for stages that immediately feed a link.

In [16]:
display_layer_evidence(4)

**Isolation case:** `nonmonotonic_batch_table` — A burst with deliberately poor size-8 decode times; table-aware grouping should prefer efficient size-4 groups over grouping every ready request.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v3-immediate-groups → v4-table-groups,650.638 → 1000.000,+349.362,+130.8%,+0.0%,-69.1%,-56.7%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 5 — SLO-aware urgency and tightly bounded waiting

This layer combines two ideas that pull in opposite directions.

### Part A: urgency

Prefill-family work uses request age relative to first-token target `SLO1`; decode-family work
uses the current token-gap clock relative to `SLO2`:

$$
u_P=\frac{\text{now}-\text{arrival}}{\text{SLO1}},\qquad
u_D=\frac{\text{now}-\text{decode clock}}{\text{SLO2}}.
$$

Only under strongly latency-weighted scoring and `u ≥ 1` can overdue work move ahead of normal
FIFO order. Small age differences do not cause constant priority churn.

In [17]:
display_source(
    source_between(layered_source, "double request_urgency", "int edge_stage_rank", 35)
)
display_table(
    [
        {"work": "prefill", "observed age": "240 ms", "target": "SLO1=300 ms", "urgency": "0.80", "overdue": "no"},
        {"work": "decode", "observed gap": "12 ms", "target": "SLO2=10 ms", "urgency": "1.20", "overdue": "yes"},
    ]
)

double request_urgency(TaskKind kind, const Request& req) const {
        const double observed = observed_request_urgency(kind, req);
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 11) {
        // SUBMISSION_FEATURE_END pre20_only
            const double predicted =
                kind == TaskKind::P_PRE || kind == TaskKind::P_POST ||
                        kind == TaskKind::P_PROC
                    ? estimated_prefill_path(kind, req) / max(1e-9, slo_tdr_)
                    : estimated_decode_path(kind, 1) / max(1e-9, slo_tpot_);
            return observed + predicted;
        // SUBMISSION_FEATURE_BEGIN pre20_only
        }
        return observed;
        // SUBMISSION_FEATURE_END pre20_only
    }

work,observed age,target,urgency,overdue
prefill,240 ms,SLO1=300 ms,0.80,no
decode,,SLO2=10 ms,1.20,yes


### Part B: controlled waiting for a better group

Waiting is allowed only when all of these are true:

- throughput weight is at least `0.95`;
- a known in-flight event will wake the scheduler;
- the table-aware target group is larger than the currently ready group;
- the oldest member has used less than half its TPOT budget; and
- elapsed waiting remains within a small `SLO2`-derived budget.

`D POST` is never deliberately held—it is already the final stage that exposes progress or
`FIN`.

In [18]:
display_source(
    source_between(
        layered_source,
        "bool should_wait_for_group",
        "bool should_defer_prefill_admission",
        85,
    )
)

bool should_wait_for_group(
        TaskKind kind,
        DurationColumn column,
        int cloud,
        int available,
        double oldest_ready_time,
        bool allow_wait
    ) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 5) {
            return false;
        }
        // SUBMISSION_FEATURE_END pre20_only
        // SUBMISSION_FEATURE_BEGIN cohort_dpost
        if constexpr (COHORT_DPOST_WAIT && kOptimizationLevel >= 20) {
            if (kind == TaskKind::D_POST) {
                if (!allow_wait || throughput_weight_ < 0.8 || available < 4) {
                    return false;
                }
                const int possible = max(total_active_decode_requests_, available);
                const int target = best_group_size(column, possible);
                if (available >= target) {
                    return false;
                }
                int future_members = 0;
                double wake_time = numeric_limits<double>::infinity();
                auto consider_future = [&](double finish_time, int members) {
                    if (finish_time + 1e-12 < wake_time) {
                        wake_time = finish_time;
                        future_members = members;
                    } else if (abs(finish_time - wake_time) <= 1e-12) {
                        future_members += members;
                    }
                };
                for (const TransferPrediction& transfer : predicted_down_queue_) {
                    if (!transfer.decode || transfer.finish_time < current_time_ - 1e-12) {
                        continue;
                    }
                    consider_future(
                        transfer.finish_time,
                        static_cast<int>(max<long long>(
                            1, transfer.size_bytes / max<long long>(1, bytes_per_token_)
                        ))
                    );
                }
                for (int future_cloud = 0; future_cloud < cloud_count_; ++future_cloud) {
                    if (!cloud_busy_[future_cloud] ||
                        cloud_running_kind_[future_cloud] != TaskKind::D_PROC ||
                        cloud_busy_until_[future_cloud] < current_time_ - 1e-12) {
                        continue;
                    }
                    const int members = max(1, cloud_running_group_size_[future_cloud]);
                    consider_future(
                        cloud_busy_until_[future_cloud] + transfer_time(
                            static_cast<long long>(members) * bytes_per_token_
                        ),
                        members
                    );
                }
                if (future_members <= 0 || !isfinite(wake_time)) {
                    return false;
                }
                double previous_duration = duration(column, 1);
                double previous_rate = 1.0 / (schedule_cost_ + previous_duration);
                for (int size = 2; size <= possible; ++size) {
                    const double next_duration = duration(column, size);
                    const double rate = static_cast<double>(size) /
                        (schedule_cost_ + next_duration);
                    if (next_duration + 1e-12 < previous_duration ||
                        rate + 1e-12 < previous_rate) {
                        return false;
                    }
                    previous_duration = next_duration;
                    previous_rate = rate;
                }
                const int merged_size = best_group_size(
                    column, min(possible, available + future_members)
                );
                if (merged_size <= available) {
                    return false;
                }
                const int remainder = merged_size - available;
                const double split_cost =
                    2.0 * schedule_cost_ + duration(column, available) +
                    duration(column, remainder);
/

### Why it can help—and why it is narrow

Urgency protects requests near a score boundary. Waiting can exchange a small amount of idle
time for a sufficiently larger group that amortizes `S`. But the protocol has event-driven
wakeups, not participant-created timers: the next event may occur later than the intended
budget. The guard is therefore much stricter than “wait whenever batching might help.”

In this layer's latency-heavy isolation scenario, controlled waiting is disabled by the
throughput-weight gate; the adjacent change is principally the urgency policy.

In [19]:
display_layer_evidence(5)

**Isolation case:** `slo_priority_collision` — One active decode stream collides with later medium prefills on a single cloud; a latency-heavy score should reward overdue decode progress.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v4-table-groups → v5-slo-aware,920.706 → 923.650,+2.944,+2.0%,+3.1%,-1.4%,-2.0%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 6 — adaptive, gap-free prefill chunks

### Bottleneck

Once a full `P PROC 0 num_layers` starts, it cannot be preempted. Decode work becoming ready
one moment later must wait for the entire prefill.

### Implemented rule

For models with more than eight layers, split `P PROC` only when the cloud has competing
decode or prefill work. Target piece duration is approximately:

$$
\max\left(4S,\min(0.25\times\text{SLO1},0.5\times\text{SLO2})\right).
$$

Convert that duration proportionally into a layer count. Pieces must be gap-free:
`[0,a)`, `[a,b)`, ..., `[z,num_layers)`.

In [20]:
display_source(
    source_between(
        layered_source,
        "int choose_prefill_piece_end",
        "vector<Candidate> cloud_candidates",
        75,
    )
)

int choose_prefill_piece_end(const Request& req, int cloud) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 6) {
            return layer_count_;
        }
        // SUBMISSION_FEATURE_END pre20_only
        const int remaining_layers = layer_count_ - req.next_prefill_layer;
        if (remaining_layers <= 1 || layer_count_ <= 8) {
            return layer_count_;
        }
        const double full_duration = duration(DurationColumn::PREFILL_PROC, req.input_length);
        const bool competing = !d_proc_ready_[cloud].empty() ||
                               active_decode_requests_[cloud] > 0 ||
                               p_proc_ready_[cloud].size() > 1;
        if (!competing) {
            return layer_count_;
        }
        const double token_multiple = layer_count_ <= 8 ? 2.0 : 0.5;
        double target_duration = max(
            4.0 * schedule_cost_,
            min(0.25 * slo_tdr_, token_multiple * slo_tpot_)
        );
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 12) {
        // SUBMISSION_FEATURE_END pre20_only
            double next_decode_milestone = next_compatible_event_time(TaskKind::D_PROC, cloud);
            for (const Request& active : requests_) {
                if (active.state == RequestState::UNSEEN ||
                    active.state == RequestState::FINISHED || active.cloud != cloud ||
                    static_cast<int>(active.state) <
                        static_cast<int>(RequestState::READY_D_PRE)) {
                    continue;
                }
                next_decode_milestone = min(
                    next_decode_milestone,
                    active.decode_clock_start + slo_tpot_
                );
            }
            if (isfinite(next_decode_milestone)) {
                const double occupied_budget = max(
                    2.0 * schedule_cost_,
                    next_decode_milestone - current_time_
                );
                target_duration = min(target_duration, occupied_budget);
            }
        // SUBMISSION_FEATURE_BEGIN pre20_only
        }
        // SUBMISSION_FEATURE_END pre20_only
        double compute_budget = target_duration;
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 12) {
        // SUBMISSION_FEATURE_END pre20_only
            compute_budget = max(
                full_duration / layer_count_,
                target_duration - schedule_cost_
            );
        // SUBMISSION_FEATURE_BEGIN pre20_only
        }
        // SUBMISSION_FEATURE_END pre20_only
        const double raw_piece_layers =
            compute_budget * layer_count_ / max(1e-12, full_duration);
        int piece_layers = 1;
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 12) {
        // SUBMISSION_FEATURE_END pre20_only
            piece_layers = static_cast<int>(floor(raw_piece_layers));
        // SUBMISSION_FEATURE_BEGIN pre20_only
        } else {
            piece_layers = static_cast<int>(ceil(raw_piece_layers));
        }
        // SUBMISSION_FEATURE_END pre20_only
        piece_layers = max(1, min(piece_layers, remaining_layers));
        return req.next_prefill_layer + piece_layers;
    }

### Estimate the chunk in the isolation scenario

In [21]:
chunk_scenario = scenario_data["single_cloud_prefill_interleave"]
chunk_profile = json.loads(
    (SCENARIO_DIR / chunk_scenario["task_times_file"]).read_text()
)["task_times"]


def interpolate_duration(rows: list[dict[str, float]], column: str, size: int) -> float:
    points = sorted(
        (int(row["batch_size"]), float(row[column]))
        for row in rows
        if float(row[column]) >= 0
    )
    if size <= points[0][0]:
        return points[0][1]
    if size >= points[-1][0]:
        return points[-1][1]
    for (left_size, left_value), (right_size, right_value) in zip(points, points[1:]):
        if size == left_size:
            return left_value
        if left_size < size < right_size:
            fraction = (size - left_size) / (right_size - left_size)
            return left_value + fraction * (right_value - left_value)
    raise AssertionError("Interpolation should have returned")


long_input = 2048
full_prefill_ms = interpolate_duration(chunk_profile, "prefill_proc", long_input)
system = chunk_scenario["system"]
scoring = chunk_scenario["scoring"]
target_ms = max(4 * system["S"], min(0.25 * scoring["SLO1"], 0.5 * scoring["SLO2"]))
piece_layers = math.ceil(target_ms * system["num_layers"] / full_prefill_ms)
display_table(
    [
        {
            "input length": long_input,
            "full P PROC ms": f"{full_prefill_ms:.2f}",
            "target piece ms": f"{target_ms:.2f}",
            "model layers": system["num_layers"],
            "estimated first piece": f"[0, {piece_layers})",
        }
    ]
)

input length,full P PROC ms,target piece ms,model layers,estimated first piece
2048,602.67,10.00,64,"[0, 2)"


### Why it can help

Every chunk boundary is a legal scheduling opportunity. The cloud can run ready `D PROC`
before continuing the next prefill piece, reducing head-of-line blocking.

### Tradeoff

Every piece pays `S`. Tiny chunks destroy throughput; giant chunks recreate the blocking
problem. The policy keeps one full piece for small models, no-competition cases, or when only
one layer remains, and targets at least `4S` of useful work per piece.

In [22]:
display_layer_evidence(6)

**Isolation case:** `single_cloud_prefill_interleave` — An active decode stream shares one cloud with a later long prefill, isolating the benefit and overhead of P PROC layer chunking.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v5-slo-aware → v6-prefill-chunks,649.866 → 673.830,+23.964,+10.2%,-33.2%,-7.7%,-9.2%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 7 — score- and collective-link-aware scheduling

### Bottleneck

Cloud compute is parallel, but every cloud shares the same FIFO `UP` queue and the same FIFO
`DOWN` queue. A huge prefill upload from one cloud can delay small latency-sensitive transfers
for all clouds.

### Implemented components

1. Add estimated transfer time to `D PRE`/`D PROC` group-size cost.
2. When latency weight exceeds throughput weight, prefer short prefill transfers within a
   bounded FIFO window; request age reduces the priority cost to resist starvation.
3. Under link pressure, favor downstream work (`D POST`, `P POST`, and `D PROC`) before adding
   more large upstream work.
4. Very narrowly defer a young prefill if an existing upload backlog already exceeds the TDR
   target and a known event will wake the scheduler.

Throughput-dominated admission preserves FIFO. Shortest-transfer-first is conditional, not a
universal scheduling law.

In [23]:
display_source(
    source_between(
        layered_source,
        "int take_link_aware_prefill_request",
        "double cloud_load_score",
        85,
    )
)
display_source(
    source_window(layered_source, "bool should_defer_prefill_admission", 30)
)

int take_link_aware_prefill_request() {
        clean_front(p_pre_ready_, RequestState::READY_P_PRE);
        if (p_pre_ready_.empty()) {
            fail("link-aware admission read an empty queue");
        }
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 7) {
            const int request_id = p_pre_ready_.front();
            p_pre_ready_.pop_front();
            return request_id;
        }
        // SUBMISSION_FEATURE_END pre20_only
        if (latency_weight_ <= throughput_weight_) {
            const int request_id = p_pre_ready_.front();
            p_pre_ready_.pop_front();
            return request_id;
        }

        const int window = min<int>(64, p_pre_ready_.size());
        int best_index = 0;
        double best_score = numeric_limits<double>::infinity();
        for (int index = 0; index < window; ++index) {
            const Request& req = request(p_pre_ready_[index]);
            const double age_ratio = (current_time_ - req.arrival_time) / max(1e-9, slo_tdr_);
            const double transfer = transfer_time(
                static_cast<long long>(req.input_length) * bytes_per_token_
            );
            const double score = transfer - age_ratio * slo_tdr_ * 0.5;
            if (score < best_score) {
                best_score = score;
                best_index = index;
            }
        }
        const int request_id = p_pre_ready_[best_index];
        p_pre_ready_.erase(p_pre_ready_.begin() + best_index);
        return request_id;
    }

bool should_defer_prefill_admission(const Request& req, bool allow_wait) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 7) {
            return false;
        }
        // SUBMISSION_FEATURE_END pre20_only
        // SUBMISSION_FEATURE_BEGIN link_admission_fairness
        if constexpr (LINK_ADMISSION_FAIRNESS) {
            bool progress_source = has_known_future_event();
            for (const deque<int>& queue : p_proc_ready_) {
                progress_source = progress_source || !queue.empty();
            }
            for (const deque<int>& queue : d_proc_ready_) {
                progress_source = progress_source || !queue.empty();
            }
            progress_source = progress_source || !p_post_ready_.empty() ||
                              !d_pre_ready_.empty() || !d_post_ready_.empty();
            const double request_transfer = transfer_time(
                static_cast<long long>(req.input_length) * bytes_per_token_
            );
            const bool earlier_pipeline = any_of(
                active_requests_.begin(), active_requests_.end(),
                [](int active) { return active > 0; }
            );
            if (progress_source && throughput_weight_ >= 0.5 &&
                request_transfer > 4.0 * slo_tpot_) {
                if (total_active_decode_requests_ > 0 || !d_pre_ready_.empty() ||
                    (total_active_decode_requests_ == 0 && earlier_pipeline &&
                     current_time_ - req.arrival_time < 0.1 * slo_tdr_)) {
                    return true;

### Why input size matters on the slow-link case

The local transfer model is:

$$
\text{transfer ms}=\text{latency ms}+\frac{8\times\text{bytes}}{\text{Gbps}\times10^6}.
$$

Compare a visible input length of 8 with 256 under this scenario's link parameters.

In [24]:
link_scenario = scenario_data["latency_weighted_slow_link"]
link_system = link_scenario["system"]


def transfer_ms(item_count: int, system: dict[str, Any]) -> float:
    size_bytes = item_count * system["bytes_per_token"]
    return system["latency_in_ms"] + 8.0 * size_bytes / (
        system["bandwidth_gbps"] * 1_000_000.0
    )


display_table(
    [
        {
            "input length": input_length,
            "bytes": input_length * link_system["bytes_per_token"],
            "estimated one-way transfer ms": f"{transfer_ms(input_length, link_system):.1f}",
        }
        for input_length in (8, 256)
    ]
)

input length,bytes,estimated one-way transfer ms
8,800000,148.0
256,25600000,4116.0


### Tradeoff

Favoring short transfers can postpone large requests. Prioritizing first-token readiness can
also worsen inter-token gaps. This scenario intentionally has high latency weight and a relaxed
TPOT target, so a large TDR improvement can outweigh a TPOT regression. Always interpret the
supplied weights before calling that trade “better.”

In [25]:
display_layer_evidence(7)

**Isolation case:** `latency_weighted_slow_link` — A slow-link burst with highly variable input sizes, a strict TDR target, and a relaxed token-gap target; shortest-prefill-first should reduce first-output readiness without pretending it is best for throughput-weighted links.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v6-prefill-chunks → v7-link-aware,777.555 → 906.241,+128.686,+0.3%,-64.2%,+312.4%,-0.3%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 8 — exact virtual timelines

Aggregate pending bytes tell us that a link is busy, but not when the next event will occur.
Layer 8 mirrors each known task finish and every FIFO transfer finish. A proposed batching wait
is allowed only when the next known wakeup fits inside the remaining wait budget. These are
conditional predictions, not clairvoyance: task and transfer service are deterministic, while
future arrivals and later policy decisions remain unknown.

In [26]:
display_source(source_between(layered_source, "void enqueue_transfer", "void complete_transfer", 80))
display_source(source_window(layered_source, "double next_known_event_time", 32))
display_layer_evidence(8)

void enqueue_transfer(
        const string& direction,
        long long size_bytes,
        int remote,
        bool decode
    ) {
        deque<TransferPrediction>* queue = nullptr;
        double* tail = nullptr;
        int* pending_count = nullptr;
        long long* pending_bytes = nullptr;
        if (direction == "UP") {
            queue = &predicted_up_queue_;
            tail = &predicted_up_tail_;
            pending_count = &pending_up_transfers_;
            pending_bytes = &pending_up_bytes_;
        } else if (direction == "DOWN") {
            queue = &predicted_down_queue_;
            tail = &predicted_down_tail_;
            pending_count = &pending_down_transfers_;
            pending_bytes = &pending_down_bytes_;
        } else {
            fail("invalid transfer direction");
        }

        const double start = max(current_time_, *tail);
        const double finish = start + transfer_time(size_bytes);
        queue->push_back({finish, size_bytes, remote, decode});
        *tail = finish;
        ++*pending_count;
        *pending_bytes += size_bytes;
    }

double next_known_event_time() const {
        double result = numeric_limits<double>::infinity();
        if (edge_busy_) {
            result = min(result, edge_busy_until_);
        }
        for (int cloud = 0; cloud < cloud_count_; ++cloud) {
            if (cloud_busy_[cloud]) {
                result = min(result, cloud_busy_until_[cloud]);
            }
        }
        if (!predicted_up_queue_.empty()) {
            result = min(result, predicted_up_queue_.front().finish_time);
        }
        if (!predicted_down_queue_.empty()) {
            result = min(result, predicted_down_queue_.front().finish_time);
        }
        return result;
    }

    bool has_known_future_event() const {
        return isfinite(next_known_event_time());
    }

    void read_arrival() {
        int request_id = 0;
        int input_length = 0;
        cin >> request_id >> input_length;
        if (request_id >= static_cast<int>(requests_.size())) {
            requests_.resize(request_id + 1);
        }
        if (requests_[request_id].state != RequestState::UNSEEN) {
            fail("duplicate ARR");

**Isolation case:** `exact_wait_horizon` — A ready decode stage sees another active stream but the next known completion is beyond its safe batching budget; exact event timing should avoid an overshooting wait.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v7-link-aware → v8-exact-timelines,281.588 → 391.776,+110.188,+34.5%,+1.1%,-28.8%,-25.6%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 9 — fanout- and cohort-aware grouping

A cross-cloud `D PRE` group creates one UP transfer per represented cloud. For the same member
count, a same-cloud group can therefore pay fewer fixed link latencies. Layer 9 enumerates FIFO
and cloud-packed candidates, retains age/urgency in the value, and predicts the serialized UP
tail. Its waiting rule also asks for a compatible cohort event—such as a decode UP completing
for this cloud—rather than treating any unrelated completion as useful.

In [27]:
display_source(source_window(layered_source, "vector<int> choose_d_pre_members", 115))
display_layer_evidence(9)

vector<int> choose_d_pre_members() {
                // SUBMISSION_FEATURE_BEGIN experimental_grouping
        if constexpr (kExperimentalGrouping) {
            const vector<int> fallback = legacy_d_pre_members();
            return choose_counterfactual_decode_group(
                d_pre_ready_,
                RequestState::READY_D_PRE,
                TaskKind::D_PRE,
                DurationColumn::DECODE_PRE,
                true,
                fallback
            );
        }
        // SUBMISSION_FEATURE_END experimental_grouping
        return legacy_d_pre_members();
    }

    bool should_wait_for_group(
        TaskKind kind,
        DurationColumn column,
        int cloud,
        int available,
        double oldest_ready_time,
        bool allow_wait
    ) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 5) {
            return false;
        }
        // SUBMISSION_FEATURE_END pre20_only
        // SUBMISSION_FEATURE_BEGIN cohort_dpost
        if constexpr (COHORT_DPOST_WAIT && kOptimizationLevel >= 20) {
            if (kind == TaskKind::D_POST) {
                if (!allow_wait || throughput_weight_ < 0.8 || available < 4) {
                    return false;
                }
                const int possible = max(total_active_decode_requests_, available);
                const int target = best_group_size(column, possible);
                if (available >= target) {
                    return false;
                }
                int future_members = 0;
                double wake_time = numeric_limits<double>::infinity();
                auto consider_future = [&](double finish_time, int members) {
                    if (finish_time + 1e-12 < wake_time) {
                        wake_time = finish_time;
                        future_members = members;
                    } else if (abs(finish_time - wake_time) <= 1e-12) {
                        future_members += members;
                    }
                };
                for (const TransferPrediction& transfer : predicted_down_queue_) {
                    if (!transfer.decode || transfer.finish_time < current_time_ - 1e-12) {
                        continue;
                    }
                    consider_future(
                        transfer.finish_time,
                        static_cast<int>(max<long long>(
                            1, transfer.size_bytes / max<long long>(1, bytes_per_token_)
                        ))
                    );
                }
                for (int future_cloud = 0; future_cloud < cloud_count_; ++future_cloud) {
                    if (!cloud_busy_[future_cloud] ||
                        cloud_running_kind_[future_cloud] != TaskKind::D_PROC ||
                        cloud_busy_until_[future_cloud] < current_time_ - 1e-12) {
                        continue;
                    }
                    const int members = max(1, cloud_running_group_size_[future_cloud]);
                    consider_future(
                        cloud_busy_until_[future_cloud] + transfer_time(
                            static_cast<long long>(members) * bytes_per_token_
                        ),
                        members
                    );
                }
                if (future_members <= 0 || !isfinite(wake_time)) {
                    return false;
                }
                double previous_duration = duration(column, 1);
                double previous_rate = 1.0 / (schedule_cost_ + previous_duration);
                for (int size = 2; size <= possible; ++size) {
                    const double next_duration = duration(column, size);
                    const double rate = static_cast<double>(size) /
                        (schedule_cost_ + next_duration);
                    if (next_duration + 1e-12 < previous_duration ||
                        rate + 1e-12 < previous_rate) {
                        return false;
             

**Isolation case:** `cross_cloud_fanout` — Eight synchronized streams span two clouds while groups of four are compute-efficient; fanout-aware D PRE selection can avoid paying two UP latencies per group.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v8-exact-timelines → v9-fanout-cohorts,991.899 → 1000.000,+8.101,+2.6%,+0.0%,+0.5%,-2.5%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 10 — batch-aware cloud placement

Cloud assignment is permanent and later `D PROC` groups must be same-cloud. Placement is thus
both load balancing and future batch formation. The policy starts from observable load and adds
a bounded batching credit only when the supplied decode curve shows an extreme per-request gain.
It may seed an existing decode cohort instead of an empty cloud, but the credit is capped so an
ordinary batching curve cannot overwhelm visible load. This is the conservative pack-versus-
spread decision.

In [28]:
display_source(source_between(layered_source, "double batch_aware_cloud_score", "int choose_cloud", 70))
display_layer_evidence(10)

double batch_aware_cloud_score(const Request& req, int cloud) const {
        (void)req;
        const int prospective_cohort = max(1, active_requests_[cloud] + 1);
        const int candidate_size = min(
            prospective_cohort,
            best_group_size(DurationColumn::DECODE_PROC, prospective_cohort)
        );
        const double singleton = schedule_cost_ + duration(DurationColumn::DECODE_PROC, 1);
        const double grouped_per_request =
            (schedule_cost_ + duration(DurationColumn::DECODE_PROC, candidate_size)) /
            candidate_size;
        const double savings_per_iteration = max(0.0, singleton - grouped_per_request);
        const double efficiency_ratio = singleton / max(1e-9, grouped_per_request);
        const double cohort_strength = min(
            4,
            active_decode_requests_[cloud] + static_cast<int>(d_proc_ready_[cloud].size())
        );
        const int minimum_active = *min_element(active_requests_.begin(), active_requests_.end());
        const bool seed_decode_cohort = minimum_active == 0 &&
                                        active_requests_[cloud] == 1 &&
                                        active_decode_requests_[cloud] > 0;
        const double credit_scale = seed_decode_cohort ? 1.0 : 0.2;
        const double credit_cap = seed_decode_cohort ? 2.0 * singleton : 0.5 * singleton;
        const double batch_credit =
            efficiency_ratio >= 3.0 &&
                    active_requests_[cloud] <= minimum_active + 1
            ? min(
                  credit_cap,
                  credit_scale * throughput_weight_ * (1.0 + 0.5 * cohort_strength) *
                      savings_per_iteration
              )
            : 0.0;
        return cloud_load_score(cloud) - batch_credit;
    }

**Isolation case:** `batch_aware_placement` — An extreme decode batching curve rewards assigning a new request to an equally loaded cloud that already has an active decode cohort.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v9-fanout-cohorts → v10-batch-placement,105.698 → 158.018,+52.320,+41.5%,+17.6%,-38.1%,-29.3%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 11 — predicted TDR/TPOT slack

Observed age alone reacts only after a request is already late. Layer 11 adds the deterministic
work still needed to reach `P POST` or the next `D POST`. A candidate is predicted overdue when

$$\text{observed age}+\widehat T_{\text{remaining path}}>\text{SLO}.$$

When two candidates are predicted overdue, the scheduler values progress toward the nearer
milestone rather than allowing a huge, already-doomed prefill to dominate merely because its
raw lateness is large. The policy remains conservative under throughput-heavy scoring.

In [29]:
display_source(source_window(layered_source, "double estimated_prefill_path", 95))
display_source(source_window(layered_source, "bool score_aware_candidate_less", 45))
display_layer_evidence(11)

double estimated_prefill_path(TaskKind kind, const Request& req) const {
        const long long bytes = static_cast<long long>(req.input_length) * bytes_per_token_;
        const double up = predicted_link_delay("UP") + transfer_time(bytes);
        const double down = predicted_link_delay("DOWN") + transfer_time(bytes);
        const double full_proc = duration(DurationColumn::PREFILL_PROC, req.input_length);
        const double remaining_fraction = layer_count_ > 0
            ? static_cast<double>(layer_count_ - req.next_prefill_layer) / layer_count_
            : 0.0;
        if (kind == TaskKind::P_PRE) {
            double best_cloud_delay = numeric_limits<double>::infinity();
            for (int cloud = 0; cloud < cloud_count_; ++cloud) {
                best_cloud_delay = min(
                    best_cloud_delay,
                    max(0.0, cloud_busy_until_[cloud] - current_time_) +
                        pending_prefill_work_[cloud]
                );
            }
            return schedule_cost_ + duration(DurationColumn::PREFILL_PRE, req.input_length) +
                   up + best_cloud_delay + schedule_cost_ + full_proc + down +
                   schedule_cost_ + duration(DurationColumn::PREFILL_POST, req.input_length);
        }
        if (kind == TaskKind::P_PROC) {
            return schedule_cost_ + remaining_fraction * full_proc + down + schedule_cost_ +
                   duration(DurationColumn::PREFILL_POST, req.input_length);
        }
        return schedule_cost_ + duration(DurationColumn::PREFILL_POST, req.input_length);
    }

    double estimated_decode_path(TaskKind kind, int group_size) const {
        const long long bytes = static_cast<long long>(group_size) * bytes_per_token_;
        const double up = predicted_link_delay("UP") + transfer_time(bytes);
        const double down = predicted_link_delay("DOWN") + transfer_time(bytes);
        const double d_pre = schedule_cost_ + duration(DurationColumn::DECODE_PRE, group_size);
        const double d_proc = schedule_cost_ + duration(DurationColumn::DECODE_PROC, group_size);
        const double d_post = schedule_cost_ + duration(DurationColumn::DECODE_POST, group_size);
        if (kind == TaskKind::D_PRE) {
            return d_pre + up + d_proc + down + d_post;
        }
        if (kind == TaskKind::D_PROC) {
            return d_proc + down + d_post;
        }
        return d_post;
    }

    double action_service_time(TaskKind kind, int group_size, const Request& req) const {
        switch (kind) {
            case TaskKind::P_PRE:
                return schedule_cost_ + duration(DurationColumn::PREFILL_PRE, req.input_length);
            case TaskKind::P_POST:
                return schedule_cost_ + duration(DurationColumn::PREFILL_POST, req.input_length);
            case TaskKind::D_PRE:
                return schedule_cost_ + duration(DurationColumn::DECODE_PRE, group_size);
            case TaskKind::D_POST:
                return schedule_cost_ + duration(DurationColumn::DECODE_POST, group_size);
            case TaskKind::P_PROC:
                return schedule_cost_ + duration(DurationColumn::PREFILL_PROC, req.input_length);
            case TaskKind::D_PROC:
                return schedule_cost_ + duration(DurationColumn::DECODE_PROC, group_size);
        }
        return 1;
    }

    double downstream_pressure(TaskKind kind, int group_size, const Request& req) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 14) {
            return 0;
        }
        // SUBMISSION_FEATURE_END pre20_only
        const double tdr_scale = max(1e-9, slo_tdr_);
        const double tpot_scale = max(1e-9, slo_tpot_);
        if (kind == TaskKind::P_PRE) {
            return predicted_link_delay("UP") / tdr_scale;
        }
        if (kind == TaskKind::P_PROC) {
            const long long bytes = static_cast<long long>(req.input_length) * bytes_per_token_;
            return (predicted_lin

bool score_aware_candidate_less(const Candidate& left, const Candidate& right) const {
        // SUBMISSION_FEATURE_BEGIN ppost_cohort_seed
        const bool left_completes_cohort = completes_small_decode_cohort(left);
        const bool right_completes_cohort = completes_small_decode_cohort(right);
        if (left_completes_cohort != right_completes_cohort) {
            return left_completes_cohort;
        }
        // SUBMISSION_FEATURE_END ppost_cohort_seed
        const bool use_deadlines = latency_weight_ > 0.8;
        const bool left_overdue = use_deadlines && left.urgency >= 1.0;
        const bool right_overdue = use_deadlines && right.urgency >= 1.0;
        if (left_overdue != right_overdue) {
            return left_overdue;
        }
        if (left_overdue) {
            const double left_priority = left.action_value +
                                         0.001 * (3 - left.stage_rank);
            const double right_priority = right.action_value +
                                          0.001 * (3 - right.stage_rank);
            if (abs(left_priority - right_priority) > 1e-12) {
                return left_priority > right_priority;
            }
        }
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 14) {
        // SUBMISSION_FEATURE_END pre20_only
            const double normalized_pressure = max(
                predicted_link_delay("UP") / max(1e-9, slo_tdr_),
                predicted_link_delay("DOWN") / max(1e-9, slo_tpot_)
            );
            // SUBMISSION_FEATURE_BEGIN pre20_only
            if constexpr (kBackpressureOlderPProc && kOptimizationLevel >= 20) {
            // SUBMISSION_FEATURE_END pre20_only
                const bool proc_pair =
                    (left.kind == TaskKind::P_PROC && right.kind == TaskKind::D_PROC) ||
                    (left.kind == TaskKind::D_PROC && right.kind == TaskKind::P_PROC);
                if (normalized_pressure > 1.0 && proc_pair &&
                    left.sequence != right.sequence) {
                    const Candidate& older = left.sequence < right.sequence ? left : right;
                    const Request& prefill_req = request(older.request_id);
                    const double prefill_service = schedule_cost_ + duration(
                        DurationColumn::PREFILL_PROC, prefill_req.input_length
                    );
                    const double decode_service = schedule_cost_ + duration(
                        DurationColumn::DECODE_PROC, 1

**Isolation case:** `predicted_deadline_slack` — A nearly complete prefill will miss SLO1 if an earlier-sequenced admission runs first; predicted remaining-path slack should prioritize P POST.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v10-batch-placement → v11-score-slack,504.908 → 537.099,+32.191,-1.1%,-4.6%,+1.2%,+1.1%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 12 — event- and deadline-aware prefill chunks

Layer 6 used a fixed SLO-derived chunk target. Layer 12 additionally finds the next compatible
decode event and the earliest active TPOT deadline on that cloud. It chooses the largest gap-free
layer range whose occupied time fits that horizon, while retaining a one-layer minimum. Since a
scheduled chunk completion creates its own event, this is legal even though the protocol has no
independent timer.

In [30]:
display_source(source_window(layered_source, "int choose_prefill_piece_end", 90))
display_layer_evidence(12)

int choose_prefill_piece_end(const Request& req, int cloud) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 6) {
            return layer_count_;
        }
        // SUBMISSION_FEATURE_END pre20_only
        const int remaining_layers = layer_count_ - req.next_prefill_layer;
        if (remaining_layers <= 1 || layer_count_ <= 8) {
            return layer_count_;
        }
        const double full_duration = duration(DurationColumn::PREFILL_PROC, req.input_length);
        const bool competing = !d_proc_ready_[cloud].empty() ||
                               active_decode_requests_[cloud] > 0 ||
                               p_proc_ready_[cloud].size() > 1;
        if (!competing) {
            return layer_count_;
        }
        const double token_multiple = layer_count_ <= 8 ? 2.0 : 0.5;
        double target_duration = max(
            4.0 * schedule_cost_,
            min(0.25 * slo_tdr_, token_multiple * slo_tpot_)
        );
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 12) {
        // SUBMISSION_FEATURE_END pre20_only
            double next_decode_milestone = next_compatible_event_time(TaskKind::D_PROC, cloud);
            for (const Request& active : requests_) {
                if (active.state == RequestState::UNSEEN ||
                    active.state == RequestState::FINISHED || active.cloud != cloud ||
                    static_cast<int>(active.state) <
                        static_cast<int>(RequestState::READY_D_PRE)) {
                    continue;
                }
                next_decode_milestone = min(
                    next_decode_milestone,
                    active.decode_clock_start + slo_tpot_
                );
            }
            if (isfinite(next_decode_milestone)) {
                const double occupied_budget = max(
                    2.0 * schedule_cost_,
                    next_decode_milestone - current_time_
                );
                target_duration = min(target_duration, occupied_budget);
            }
        // SUBMISSION_FEATURE_BEGIN pre20_only
        }
        // SUBMISSION_FEATURE_END pre20_only
        double compute_budget = target_duration;
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 12) {
        // SUBMISSION_FEATURE_END pre20_only
            compute_budget = max(
                full_duration / layer_count_,
                target_duration - schedule_cost_
            );
        // SUBMISSION_FEATURE_BEGIN pre20_only
        }
        // SUBMISSION_FEATURE_END pre20_only
        const double raw_piece_layers =
            compute_budget * layer_count_ / max(1e-12, full_duration);
        int piece_layers = 1;
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 12) {
        // SUBMISSION_FEATURE_END pre20_only
            piece_layers = static_cast<int>(floor(raw_piece_layers));
        // SUBMISSION_FEATURE_BEGIN pre20_only
        } else {
            piece_layers = static_cast<int>(ceil(raw_piece_layers));
        }
        // SUBMISSION_FEATURE_END pre20_only
        piece_layers = max(1, min(piece_layers, remaining_layers));
        return req.next_prefill_layer + piece_layers;
    }

    vector<Candidate> cloud_candidates(int cloud) {
        vector<Candidate> candidates;
        auto add = [&](TaskKind kind, deque<int>& queue, RequestState expected, int rank) {
            if (!queue_available(queue, expected)) {
                return;
            }
            const int request_id = queue.front();
            const Request& req = request(request_id);
            const int group_size = kind == TaskKind::D_PROC
                ? best_group_size(DurationColumn::DECODE_PROC, queue.size())
                : 1;
            candidates.push_back(
                {kind,
                 request_id,
                 req.ready_sequence,

**Isolation case:** `chunk_deadline_collision` — A long prefill shares one cloud with active decode and exposes whether chunk duration is tied to the next decode deadline rather than a fixed fraction of the SLOs.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v11-score-slack → v12-deadline-chunks,764.239 → 780.814,+16.575,+0.0%,+0.1%,-31.5%,+0.0%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 13 — attained-service and online survival estimates

Total output length is hidden, but tokens already produced are observable. With little history,
layer 13 uses a least-attained-service/MLFQ-style preference plus aging. Every `FIN` supplies one
completed output length; after enough samples, the scheduler estimates expected remaining tokens
among completed requests that survived at least as long, first within an input-length bin and then
globally. SLO urgency and aging prevent a long stream from starving indefinitely.

In [31]:
display_source(source_window(layered_source, "double expected_remaining_tokens", 75))
display_source(source_window(layered_source, "vector<int> take_decode_members", 45))
display_layer_evidence(13)

double expected_remaining_tokens(const Request& req) const {
        auto survivor_mean = [&](const vector<int>& samples, int minimum_samples)
            -> optional<double> {
            double sum = 0;
            int count = 0;
            for (int length : samples) {
                if (length > req.produced_tokens) {
                    sum += length - req.produced_tokens;
                    ++count;
                }
            }
            if (count < minimum_samples) {
                return nullopt;
            }
            return sum / count;
        };

        const vector<int>& local_samples =
            completed_output_lengths_by_input_bin_[input_length_bin(req.input_length)];
        if (optional<double> estimate = survivor_mean(local_samples, 3)) {
            return *estimate;
        }
        if (optional<double> estimate = survivor_mean(completed_output_lengths_, 5)) {
            return *estimate;
        }
        // Least-attained-service fallback when there is not enough completed history to learn
        // a survival curve. It deliberately gives new streams a short-job opportunity.
        return 1.0 + req.produced_tokens;
    }

    // SUBMISSION_FEATURE_BEGIN censored_completion_index
    double empirical_completion_index(const Request& req) const {
        const int input_bin = input_length_bin(req.input_length);
        const int next_age = min(kCompletionAgeBuckets - 1, req.produced_tokens + 1);
        if (completion_reached_[next_age] < CENSORED_COMPLETION_MIN_REACHED) {
            return 1.0 / max(1.0, expected_remaining_tokens(req));
        }
        double survival = 1.0;
        double finish_probability = 0.0;
        double expected_service = 0.0;
        double best_index = 0.0;
        for (int quantum = 1; quantum <= 16; ++quantum) {
            const int age = min(
                kCompletionAgeBuckets - 1, req.produced_tokens + quantum
            );
            int reached = completion_reached_by_input_[input_bin][age];
            int finished = completion_finished_by_input_[input_bin][age];
            if (reached < max(6, CENSORED_COMPLETION_MIN_REACHED / 2)) {
                reached = completion_reached_[age];
                finished = completion_finished_[age];
            }
            const double hazard = (finished + 1.0) / (reached + 9.0);
            expected_service += survival;
            finish_probability += survival * hazard;
            survival *= 1.0 - hazard;
            best_index = max(
                best_index,
                finish_probability / max(1e-12, expected_service)
            );
        }
        return best_index;
    }
    // SUBMISSION_FEATURE_END censored_completion_index

    double estimated_prefill_path(TaskKind kind, const Request& req) const {
        const long long bytes = static_cast<long long>(req.input_length) * bytes_per_token_;
        const double up = predicted_link_delay("UP") + transfer_time(bytes);
        const double down = predicted_link_delay("DOWN") + transfer_time(bytes);
        const double full_proc = duration(DurationColumn::PREFILL_PROC, req.input_length);
        const double remaining_fraction = layer_count_ > 0
            ? static_cast<double>(layer_count_ - req.next_prefill_layer) / layer_count_
            : 0.0;
        if (kind == TaskKind::P_PRE) {
            double best_cloud_delay = numeric_limits<double>::infinity();
            for (int cloud = 0; cloud < cloud_count_; ++cloud) {

vector<int> take_decode_members(
        deque<int>& queue,
        RequestState expected,
        int count,
        TaskKind kind
    ) {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 13) {
            return take_front(queue, expected, count);
        }
        // SUBMISSION_FEATURE_END pre20_only
        vector<int> candidates = collect_ready(queue, expected);
        stable_sort(candidates.begin(), candidates.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), kind);
            const double right_value = decode_member_value(request(right), kind);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });
        candidates.resize(min<int>(count, candidates.size()));
        return take_selected(queue, expected, candidates);
    }

    int take_link_aware_prefill_request() {
        clean_front(p_pre_ready_, RequestState::READY_P_PRE);
        if (p_pre_ready_.empty()) {
            fail("link-aware admission read an empty queue");
        }
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 7) {
            const int request_id = p_pre_ready_.front();
            p_pre_ready_.pop_front();
            return request_id;
        }
        // SUBMISSION_FEATURE_END pre20_only
        if (latency_weight_ <= throughput_weight_) {
            const int request_id = p_pre_ready_.front();
            p_pre_ready_.pop_front();
            return request_id;
        }

        const int window = min<int>(64, p_pre_ready_.size());
        int best_index = 0;
        double best_score = numeric_limits<double>::infinity();

**Isolation case:** `attained_service_tail` — One long-lived stream competes with later short streams under a group-size-two optimum, exposing least-attained-service and learned-remaining selection.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v12-deadline-chunks → v13-attained-service,902.231 → 903.701,+1.470,+0.6%,+0.0%,-1.4%,-0.6%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 14 — downstream link backpressure

Starting legal work is not always useful if it injects another transfer into a saturated FIFO
queue. Layer 14 measures the predicted UP and DOWN tails relative to SLO scales and subtracts a
downstream-pressure penalty from actions that add work. Under severe pressure this can move an
exit task ahead of a new admission. The isolation run is intentionally honest: the dispatch order
changes, but its aggregate score is neutral, showing that queue-theoretic pressure is not by
itself the contest objective.

In [32]:
display_source(source_window(layered_source, "double downstream_pressure", 52))
display_layer_evidence(14)

double downstream_pressure(TaskKind kind, int group_size, const Request& req) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 14) {
            return 0;
        }
        // SUBMISSION_FEATURE_END pre20_only
        const double tdr_scale = max(1e-9, slo_tdr_);
        const double tpot_scale = max(1e-9, slo_tpot_);
        if (kind == TaskKind::P_PRE) {
            return predicted_link_delay("UP") / tdr_scale;
        }
        if (kind == TaskKind::P_PROC) {
            const long long bytes = static_cast<long long>(req.input_length) * bytes_per_token_;
            return (predicted_link_delay("DOWN") + transfer_time(bytes)) / tdr_scale;
        }
        if (kind == TaskKind::D_PRE) {
            return predicted_link_delay("UP") / tpot_scale;
        }
        if (kind == TaskKind::D_PROC) {
            const long long bytes = static_cast<long long>(group_size) * bytes_per_token_;
            return (predicted_link_delay("DOWN") + transfer_time(bytes)) / tpot_scale;
        }
        return 0;
    }

    double action_value(TaskKind kind, const Request& req, int group_size) const {
        const double service = max(1e-9, action_service_time(kind, group_size, req));
        const double urgency = request_urgency(kind, req);
        double progress = 0.4;
        if (kind == TaskKind::P_POST || kind == TaskKind::D_POST) {
            progress = 1.5;
        } else if (kind == TaskKind::P_PROC || kind == TaskKind::D_PROC) {
            progress = 1.0;
        } else if (kind == TaskKind::D_PRE) {
            progress = 0.75;
        }
        const double milestone =
            kind == TaskKind::P_PRE || kind == TaskKind::P_POST || kind == TaskKind::P_PROC
                ? estimated_prefill_path(kind, req)
                : estimated_decode_path(kind, group_size);
        const double latency_value =
            min(2.0, max(0.0, urgency - 0.5)) * progress / max(1e-9, milestone);
        const double throughput_value = static_cast<double>(group_size) / service;
        const double pressure = downstream_pressure(kind, group_size, req);
        double value = latency_weight_ * latency_value +
                       throughput_weight_ * throughput_value - 0.2 * pressure;
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel >= 15) {
        // SUBMISSION_FEATURE_END pre20_only
            value += 0.25 * (latency_weight_ + 0.25 * throughput_weight_) /
                     max(1e-9, milestone);
        // SUBMISSION_FEATURE_BEGIN pre20_only

**Isolation case:** `downstream_backpressure` — A saturated UP queue coincides with both a new prefill admission and ready token completion; backpressure should drain D POST before injecting more work.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v13-attained-service → v14-backpressure,10.257 → 10.257,+0.000,+0.0%,+0.0%,+0.1%,+0.0%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 15 — bounded one-token lookahead

A group can look fast at `D PROC` yet be disastrous at its downstream `D POST`. Layer 15 detects
a hostile downstream curve when a group's per-member post cost is more than 1.5 times the best
smaller choice. Only then does it extend group evaluation through the remaining one-token path.
This bounded trigger preserves ordinary fanout decisions and avoids a combinatorial search on up
to two million frames. The scheduler executes one action and replans at the next event.

In [33]:
display_source(source_window(layered_source, "bool downstream_group_is_hostile", 35))
display_source(source_window(layered_source, "if constexpr (kOptimizationLevel >= 15)", 42))
display_layer_evidence(15)

bool downstream_group_is_hostile(int size) const {
        if (size <= 1) {
            return false;
        }
        double best_per_member = numeric_limits<double>::infinity();
        for (int candidate = 1; candidate <= size; ++candidate) {
            best_per_member = min(
                best_per_member,
                (schedule_cost_ + duration(DurationColumn::DECODE_POST, candidate)) /
                    candidate
            );
        }
        const double current_per_member =
            (schedule_cost_ + duration(DurationColumn::DECODE_POST, size)) / size;
        return current_per_member > 1.5 * best_per_member + 1e-12;
    }

    double next_compatible_event_time(TaskKind kind, int cloud) const {
        double result = numeric_limits<double>::infinity();
        if (kind == TaskKind::D_PRE && edge_busy_ &&
            edge_running_kind_ == TaskKind::D_POST) {
            result = min(result, edge_busy_until_);
        }
        if (kind == TaskKind::D_PROC) {
            for (const TransferPrediction& transfer : predicted_up_queue_) {
                if (transfer.decode && transfer.remote == cloud) {
                    result = min(result, transfer.finish_time);
                    break;
                }
            }
        }
        return result;
    }

    double expected_remaining_tokens(const Request& req) const {

if constexpr (kOptimizationLevel >= 15) {
        // SUBMISSION_FEATURE_END pre20_only
            value += 0.25 * (latency_weight_ + 0.25 * throughput_weight_) /
                     max(1e-9, milestone);
        // SUBMISSION_FEATURE_BEGIN pre20_only
        }
        // SUBMISSION_FEATURE_END pre20_only
        return value;
    }

    double decode_member_value(const Request& req, TaskKind kind) const {
        const double urgency = request_urgency(kind, req);
        const double waited = max(0.0, current_time_ - req.ready_time);
        const double aging = min(2.0, waited / max(1e-9, 2.0 * slo_tpot_));
        const double learned_shortness = 1.0 / max(1.0, expected_remaining_tokens(req));
        int level = 0;
        for (int tokens = req.produced_tokens + 1; tokens > 1; tokens >>= 1) {
            ++level;
        }
        // SUBMISSION_FEATURE_BEGIN censored_completion_index
        if constexpr (CENSORED_COMPLETION_INDEX) {
            return 4.0 * max(0.0, urgency - 1.0) + urgency + aging +
                   empirical_completion_index(req) - 0.2 * level;
        }
        // SUBMISSION_FEATURE_END censored_completion_index
        return 4.0 * max(0.0, urgency - 1.0) + urgency + aging + learned_shortness -
               0.2 * level;
    }

    vector<int> take_decode_members(
        deque<int>& queue,
        RequestState expected,
        int count,
        TaskKind kind
    ) {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (kOptimizationLevel < 13) {
            return take_front(queue, expected, count);
        }
        // SUBMISSION_FEATURE_END pre20_only
        vector<int> candidates = collect_ready(queue, expected);
        stable_sort(candidates.begin(), candidates.end(), [&](int left, int right) {

**Isolation case:** `one_token_lookahead` — Local D PROC throughput favors four members, but the downstream D POST curve makes groups of two faster end to end; one-token lookahead should see the complete path.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v14-backpressure → v15-one-token-lookahead,604.412 → 729.520,+125.108,+20.2%,+0.0%,-15.8%,-16.8%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 16 — bounded counterfactual grouping

Layers 4 and 15 mainly start from a group size; layer 16 explicitly constructs alternative
memberships and asks what each legal action is predicted to do. Candidate sizes are bounded to
the table-rate optimum, fractions of the ready set, and nearby table breakpoints. Membership
variants include FIFO, urgency, attained-service value, and same-cloud packing.

For each candidate, the policy rolls a virtual one-token path through the edge, FIFO UP queue,
same-cloud processing cohorts, FIFO DOWN queue, and edge post-processing. Its feature vector
includes normalized token rate, schedule-cost amortization, predicted TPOT quality, urgency,
completion potential, excluded-request pressure, cloud fanout, finish dispersion, and link
pressure. It keeps the v15 action unless the new value clears a safety margin, and it never
increases D PRE cloud fanout relative to that fallback.

In [34]:
display_source(source_window(layered_source, "vector<int> bounded_candidate_group_sizes", 48))
display_source(source_window(layered_source, "GroupEvaluation evaluate_decode_group", 95))
display_source(source_window(layered_source, "vector<int> choose_counterfactual_decode_group", 105))
display_layer_evidence(16)

vector<int> bounded_candidate_group_sizes(
        DurationColumn column,
        int available
    ) const {
        set<int> sizes = {1, available};
        const int best = best_group_size(column, available);
        for (int candidate : {
                 best - 1,
                 best,
                 best + 1,
                 best / 2,
                 min(available, 2 * best),
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (1 <= candidate && candidate <= available) {
                sizes.insert(candidate);
            }
        }

        const vector<pair<int, double>>& curve =
            duration_curves_[static_cast<int>(column)];
        auto near_best = lower_bound(
            curve.begin(), curve.end(), make_pair(best, -numeric_limits<double>::infinity())
        );
        for (int offset = -2; offset <= 2; ++offset) {
            const long long index = distance(curve.begin(), near_best) + offset;
            if (0 <= index && index < static_cast<long long>(curve.size())) {
                const int candidate = min(available, curve[index].first);
                if (candidate >= 1) {
                    sizes.insert(candidate);
                }
            }
        }
        return vector<int>(sizes.begin(), sizes.end());
    }

    GroupEvaluation evaluate_decode_group(
        TaskKind kind,
        DurationColumn column,
        const vector<int>& group,
        const vector<int>& all_ready
    ) const {
        GroupEvaluation result;
        const int size = static_cast<int>(group.size());
        vector<int> counts(cloud_count_, 0);
        for (int request_id : group) {

GroupEvaluation evaluate_decode_group(
        TaskKind kind,
        DurationColumn column,
        const vector<int>& group,
        const vector<int>& all_ready
    ) const {
        GroupEvaluation result;
        const int size = static_cast<int>(group.size());
        vector<int> counts(cloud_count_, 0);
        for (int request_id : group) {
            ++counts[request(request_id).cloud];
        }
        result.fanout = count_if(counts.begin(), counts.end(), [](int count) {
            return count > 0;
        });

        const double stage_service = schedule_cost_ + duration(column, size);
        double earliest_down_finish = numeric_limits<double>::infinity();
        double latest_down_finish = current_time_;

        if (kind == TaskKind::D_POST) {
            result.token_finish = current_time_ + stage_service;
        } else if (kind == TaskKind::D_PROC) {
            const double process_finish = current_time_ + stage_service;
            const double down_finish = max(process_finish, predicted_down_tail_) +
                                       transfer_time(
                                           static_cast<long long>(size) * bytes_per_token_
                                       );
            earliest_down_finish = latest_down_finish = down_finish;
            result.token_finish = max(down_finish, edge_busy_until_) + schedule_cost_ +
                                  duration(DurationColumn::DECODE_POST, size);
        } else {
            const double edge_finish = current_time_ + stage_service;
            double up_tail = max(edge_finish, predicted_up_tail_);
            vector<pair<double, int>> process_cohorts;
            for (int cloud = 0; cloud < cloud_count_; ++cloud) {
                if (counts[cloud] == 0) {
                    continue;
                }
                up_tail += transfer_time(
                    static_cast<long long>(counts[cloud]) * bytes_per_token_
                );
                const double process_finish =
                    max(up_tail, cloud_busy_until_[cloud]) + schedule_cost_ +
                    duration(DurationColumn::DECODE_PROC, counts[cloud]);
                process_cohorts.push_back({process_finish, cloud});
            }
            sort(process_cohorts.begin(), process_cohorts.end());

            double down_tail = max(current_time_, predicted_down_tail_);
            for (const auto& [process_finish, cloud] : process_cohorts) {
                down_tail = max(down_tail, process_finish) + transfer_time(
                    static_cast<long long>(counts[cloud]) * bytes_per_token_
                );
                earliest_down_finish = min(earliest_down_finish, down_tail);
                latest_down_finish = max(latest_down_finish, down_tail);
            }
            result.token_finish = max(down_tail, edge_finish) + schedule_cost_ +
                                  duration(DurationColumn::DECODE_POST, size);
        }

        result.horizon = max(1e-9, result.token_finish - current_time_);
        const double rate = size / result.horizon;
        const double rate_reference = max(
            1e-9,
            max(throughput_upper_bound_, throughput_baseline_)
        );
        result.normalized_rate = min(4.0, rate / rate_reference);

        const double singleton_service = schedule_cost_ + duration(column, 1);
        result.service_efficiency = max(
            -2.0,
            min(
                1.0,
                1.0 - stage_service / max(1e-9, size * singleton_service)
            )
        );

        double gap_sum = 0;
        double urgency_sum = 0;
        double completion_sum = 0;
        for (int request_id : group) {
            const Request& req = request(request_id);
            gap_sum += max(0.0, result.token_finish - req.decode_clock_start);
            urgency_sum += request_urgency(kind, req);
            completion_sum += 1.0 / max(1.0, expected_remaining_tokens(req));
        }
        const doub

vector<int> choose_counterfactual_decode_group(
        deque<int>& queue,
        RequestState expected,
        TaskKind kind,
        DurationColumn column,
        bool add_cloud_packing,
        const vector<int>& fallback_group
    ) {
        vector<int> ready = collect_ready(queue, expected);
        if (ready.empty()) {
            fail("counterfactual grouping found no ready members");
        }
        if (ready.size() == 1) {
            return ready;
        }

        vector<int> by_urgency = ready;
        stable_sort(by_urgency.begin(), by_urgency.end(), [&](int left, int right) {
            const double left_value = request_urgency(kind, request(left));
            const double right_value = request_urgency(kind, request(right));
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        vector<int> by_member_value = ready;
        stable_sort(by_member_value.begin(), by_member_value.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), kind);
            const double right_value = decode_member_value(request(right), kind);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        map<int, vector<int>> by_cloud;
        if (add_cloud_packing) {
            for (int request_id : by_member_value) {
                by_cloud[request(request_id).cloud].push_back(request_id);
            }
        }

        vector<vector<int>> candidates;
        const vector<int> sizes = bounded_candidate_group_sizes(column, ready.size());
        auto add_prefix = [&](const vector<int>& order, int size) {
            candidates.emplace_back(order.begin(), order.begin() + size);
        };
        for (int size : sizes) {
            add_prefix(ready, size);
            add_prefix(by_urgency, size);
            add_prefix(by_member_value, size);

            if (!add_cloud_packing) {
                continue;
            }
            for (const auto& [anchor_cloud, anchor_members] : by_cloud) {
                vector<int> packed;
                packed.reserve(size);
                for (int request_id : anchor_members) {
                    if (static_cast<int>(packed.size()) == size) {
                        break;
                    }
                    packed.push_back(request_id);
                }
                vector<pair<int, int>> remaining_clouds;
                for (const auto& [cloud, members] : by_cloud) {
                    if (cloud != anchor_cloud) {
                        remaining_clouds.push_back(
                            {-static_cast<int>(members.size()), cloud}
                        );
                    }
                }
                sort(remaining_clouds.begin(), remaining_clouds.end());
                for (const auto& [ignored_size, cloud] : remaining_clouds) {
                    (void)ignored_size;
                    for (int request_id : by_cloud[cloud]) {
                        if (static_cast<int>(packed.size()) == size) {
                            break;
                        }
                        packed.push_back(request_id);
                    }
                    if (static_cast<int>(packed.size()) == size) {
                        break;
                    }
                }
                if (static_cast<int>(packed.size()) == size) {
                    candidates.push_back(std::move(packed));
                }
            }
        }

        vector<int> best_group = fallback_group;
        if (best_group.empty()) {
            best_group = {ready.front()};
        }
        GroupEvaluation best_evaluation = evaluate_decode_group(
            kind, column, best_group, ready
        )

**Isolation case:** `counterfactual_grouping` — A frozen generated workload where bounded end-to-end group evaluation beats the independent stage-rate choice; promoted from the training generator for layer-16 mechanism evidence.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v15-one-token-lookahead → v16-counterfactual-groups,356.998 → 360.418,+3.420,+8.9%,+0.0%,-17.8%,-8.2%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

### Why this layer is still experimental

One-token prediction is conditional on current queues and cannot know future arrivals or hidden
output lengths. It also approximates how independently completing cloud cohorts will regroup at
`D POST`. The isolation case shows a real local gain, but other generated cases regress sharply;
the full validation decision must therefore include train/holdout behavior, not this one win.

## Layer 17 — offline-fitted conservative ranker

Layer 17 keeps the same legal candidate generator and deterministic simulator but replaces its
hand-set coefficients with values selected by black-box policy search. The generator creates 18
training and 12 holdout workloads across balanced, edge-amortized, cloud-amortized, slow-link,
post-hostile, and latency-heavy families. Search maximizes a regression-penalized objective on
training only; holdout is opened after selection.

Hidden output lengths appear only inside the local interactor when it calculates the final label.
They are never features available to the submitted scheduler. The exported C++ ranker is a few
constants and arithmetic operations, so it needs no model file or Python runtime.

In [35]:
tuning_report = json.loads(TUNING_REPORT_PATH.read_text())
v17_audit = tuning_report["selected"]["v17"]
display_table(
    [
        {
            "split": split,
            "mean score delta vs v15": f"{v17_audit[split]['mean_delta_vs_v15']:+.3f}",
            "wins / ties / losses": (
                f"{v17_audit[split]['wins']} / {v17_audit[split]['ties']} / "
                f"{v17_audit[split]['losses']}"
            ),
            "worst delta": f"{v17_audit[split]['worst_delta']:+.3f}",
        }
        for split in ("train", "holdout")
    ]
)
display_source(source_window(layered_source, "double counterfactual_group_value", 65))
display_layer_evidence(17)

split,mean score delta vs v15,wins / ties / losses,worst delta
train,+0.044,1 / 17 / 0,+0.000
holdout,-0.345,0 / 11 / 1,-4.144


double counterfactual_group_value(const GroupEvaluation& group) const {
        double rate_weight = 1.15;
        double efficiency_weight = 0.40;
        double waiting_weight = 1.00;
        double urgency_weight = 0.40;
        double completion_weight = 0.10;
        double fanout_penalty = 0.20;
        double excluded_penalty = 0.28;
        double dispersion_penalty = 0.15;
        if constexpr (kOptimizationLevel >= 17) {
            rate_weight = GROUP_RATE_WEIGHT;
            efficiency_weight = GROUP_EFFICIENCY_WEIGHT;
            waiting_weight = GROUP_LATENCY_WEIGHT;
            urgency_weight = GROUP_URGENCY_WEIGHT;
            completion_weight = GROUP_COMPLETION_WEIGHT;
            fanout_penalty = GROUP_FANOUT_PENALTY;
            excluded_penalty = GROUP_EXCLUDED_PENALTY;
            dispersion_penalty = GROUP_DISPERSION_PENALTY;
        }

        double value =
            throughput_weight_ *
                (rate_weight * group.normalized_rate +
                 efficiency_weight * group.service_efficiency) +
            latency_weight_ *
                (waiting_weight * group.waiting_quality +
                 urgency_weight * group.urgency_progress +
                 completion_weight * group.completion_potential) -
            fanout_penalty * group.fanout_pressure -
            excluded_penalty * group.excluded_pressure -
            dispersion_penalty * group.finish_dispersion;

        if constexpr (kOptimizationLevel >= 18) {
            const double uncongested = max(0.0, 1.0 - group.link_pressure);
            value += GROUP_INTERACTION_EFFICIENCY * throughput_weight_ *
                     group.service_efficiency * uncongested;
            value += GROUP_INTERACTION_URGENCY * latency_weight_ *
                     group.urgency_progress * (1.0 - group.waiting_quality);
            value -= GROUP_INTERACTION_CONGESTION *
                     (group.fanout_pressure * group.link_pressure +
                      group.excluded_pressure * max(0.0, group.urgency_progress - 1.0));
        }
        return value;
    }

    vector<int> choose_counterfactual_decode_group(
        deque<int>& queue,
        RequestState expected,
        TaskKind kind,
        DurationColumn column,
        bool add_cloud_packing,
        const vector<int>& fallback_group
    ) {
        vector<int> ready = collect_ready(queue, expected);
        if (ready.empty()) {
            fail("counterfactual grouping found no ready members");
        }
        if (ready.size() == 1) {
            return ready;
        }

        vector<int> by_urgency = ready;
        stable_sort(by_urgency.begin(), by_urgency.end(), [&](int left, int right) {
            const double left_value = request_urgency(kind, request(left));
            const double right_value = request_urgency(kind, request(right));

**Isolation case:** `learned_grouping_recovery` — A frozen generated post-hostile workload where untuned counterfactual scoring regresses and the offline-fitted conservative ranker recovers the proven v15 behavior.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v16-counterfactual-groups → v17-learned-group-ranker,791.481 → 917.676,+126.196,+9.0%,+0.0%,-19.2%,-8.3%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

### What the audit says

The selected linear ranker improved one of 18 training scenarios and tied the rest, but lost
4.144 points on one of 12 holdout scenarios. It exactly matched v15 on the original checked-in
suite before the three learned-policy audit cases were promoted. That is useful learning evidence,
but not enough to replace the current submission.

## Layer 18 — test nonlinear interactions, then accept the null result

Layer 18 offered three extra interactions: efficiency when the link is uncongested, urgency when
predicted waiting quality is poor, and a congestion interaction between fanout/link pressure and
excluded-request pressure. The same train-only selection procedure tested these terms.

The winning candidate set all three interaction weights to exactly zero. Consequently v18 is
behaviorally identical to v17. Keeping this zero-delta layer in the registry is intentional: it
records that the tested added complexity was rejected instead of presenting an unvalidated model
as an optimization.

In [36]:
v18_audit = tuning_report["selected"]["v18"]
display_table(
    [
        {
            "interaction": name,
            "selected weight": f"{v18_audit['weights'][name]:.3f}",
        }
        for name in (
            "GROUP_INTERACTION_EFFICIENCY",
            "GROUP_INTERACTION_URGENCY",
            "GROUP_INTERACTION_CONGESTION",
        )
    ]
)
display_source(source_window(layered_source, "if constexpr (kOptimizationLevel >= 18)", 16))
display_layer_evidence(18)

interaction,selected weight
GROUP_INTERACTION_EFFICIENCY,0.000
GROUP_INTERACTION_URGENCY,0.000
GROUP_INTERACTION_CONGESTION,0.000


if constexpr (kOptimizationLevel >= 18) {
            const double uncongested = max(0.0, 1.0 - group.link_pressure);
            value += GROUP_INTERACTION_EFFICIENCY * throughput_weight_ *
                     group.service_efficiency * uncongested;
            value += GROUP_INTERACTION_URGENCY * latency_weight_ *
                     group.urgency_progress * (1.0 - group.waiting_quality);
            value -= GROUP_INTERACTION_CONGESTION *
                     (group.fanout_pressure * group.link_pressure +
                      group.excluded_pressure * max(0.0, group.urgency_progress - 1.0));
        }
        return value;
    }

    vector<int> choose_counterfactual_decode_group(
        deque<int>& queue,
        RequestState expected,

**Isolation case:** `nonlinear_ranker_holdout` — A frozen formerly held-out post-hostile workload used to show that the selected nonlinear interaction layer is a no-op because training rejected every higher-complexity candidate.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v17-learned-group-ranker → v18-nonlinear-group-ranker,332.199 → 332.199,+0.000,+0.0%,+0.0%,+0.0%,+0.0%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

## Layer 19 — optimize the finite D POST remainder, conservatively

Layer 15 chooses a D POST size largely from the steady-state rate

\[
R(g)=\frac{g}{S+T_{D\ POST}(g)}.
\]

That is the right question for an indefinitely replenished queue, but a finite ready queue can
have a different optimum. If choosing `g` leaves a small remainder whose next batch pays another
large scheduling and post-processing cost, then the first group with the best local rate can
produce a worse total clearance time. The relevant short-horizon quantity is

\[
C_n(g)=S+T_{D\ POST}(g)+C_{n-g},
\]

where `n` is the currently known queue and subsequent groups return to the v15 size rule. Layer
19 therefore simulates the entire known terminal queue for a bounded set of first-group sizes.
It also inserts decode DOWN transfers whose completion times are already known, because ignoring
one incoming cohort caused a large adversarial regression: an oversized current batch blocked a
much more efficient combined batch a moment later.

The implementation is intentionally asymmetric and conservative:

- it starts from the v15 group and only considers a **larger** first group;
- it uses only observed TDR/TPOT, current queues, supplied task times, and scheduled DOWN arrivals;
- the candidate must improve modeled queue clearance by at least 2% and improve the score surrogate;
- otherwise it emits the exact v15 fallback.

This is a terminal-stage heuristic, not knowledge of hidden output lengths or unscheduled future
arrivals. It branches from v15, so the comparison below is `v15 → v19`, not `v18 → v19`.

In [37]:
display_source(source_window(layered_source, "vector<int> terminal_dpost_members", 178))
display_layer_evidence(19)

vector<int> terminal_dpost_members() {
        vector<int> ready = collect_ready(d_post_ready_, RequestState::READY_D_POST);
        if (ready.empty()) {
            fail("terminal D POST selection found no ready members");
        }
        stable_sort(ready.begin(), ready.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), TaskKind::D_POST);
            const double right_value = decode_member_value(request(right), TaskKind::D_POST);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        const int available = static_cast<int>(ready.size());
        const int fallback_size = best_group_size(
            DurationColumn::DECODE_POST, available
        );
        // SUBMISSION_FEATURE_BEGIN exact_dpost_partition
        const int partition_size = EXACT_DPOST_PARTITION
            ? exact_dpost_partition_first(ready)
            : fallback_size;
        // SUBMISSION_FEATURE_END exact_dpost_partition
        auto prefix = [&](int size) {
            return vector<int>(ready.begin(), ready.begin() + size);
        };
        if (available <= 1 || available > 96 || fallback_size <= 1 ||
            latency_weight_ <= 0.1 || distance_baseline_ <= 0) {
            return prefix(fallback_size);
        }

        set<int> sizes = {1, fallback_size, available};
        // SUBMISSION_FEATURE_BEGIN exact_dpost_partition
        sizes.insert(partition_size);
        // SUBMISSION_FEATURE_END exact_dpost_partition
        for (int size : {
                 fallback_size - 1,
                 fallback_size + 1,
                 fallback_size / 2,
                 min(available, 2 * fallback_size),
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (1 <= size && size <= available) {
                sizes.insert(size);
            }
        }
        const vector<pair<int, double>>& curve =
            duration_curves_[static_cast<int>(DurationColumn::DECODE_POST)];
        for (int anchor : {fallback_size, available / 2}) {
            auto position = lower_bound(
                curve.begin(), curve.end(),
                make_pair(anchor, -numeric_limits<double>::infinity())
            );
            for (int offset = -2; offset <= 2; ++offset) {
                const long long index = distance(curve.begin(), position) + offset;
                if (0 <= index && index < static_cast<long long>(curve.size())) {
                    sizes.insert(min(available, curve[index].first));
                }
            }
        }

        vector<pair<double, int>> future_arrivals;
        int future_members = 0;
        for (const TransferPrediction& transfer : predicted_down_queue_) {
            if (!transfer.decode || transfer.finish_time < current_time_ - 1e-12 ||
                future_arrivals.size() >= 8 || future_members >= 96 - available) {
                continue;
            }
            const int members = min<int>(
                max<long long>(1, transfer.size_bytes / max<long long>(1, bytes_per_token_)),
                96 - available - future_members
            );
            if (members > 0) {
                future_arrivals.push_back({transfer.finish_time, members});
                future_members += members;
            }
        }

        auto value = [&](int first_size) {
            int completed_ready = 0;
            int queued = available;
            size_t arrival_index = 0;
            bool first_group = true;
            double virtual_time = current_time_;
            double gap_sum = observed_tpot_sum_;
            long long gap_count = observed_tpot_count_;
            while (queued > 0 || arrival_index < future_arrivals.size()) {
                if (queued == 0) {
                    virtual_time = max(
          

**Isolation case:** `terminal_dpost_remainder` — Finite D POST queue where the steady-state-rate batch leaves an expensive remainder and a larger first group clears the terminal queue faster.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v15-one-token-lookahead → v19-terminal-dpost,448.898 → 457.520,+8.622,+1.7%,+0.0%,-2.9%,-1.7%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

### Adversarial search and the untouched audit split

Random-looking average workloads are weak tests for a batching heuristic. The adversarial
generator varies D POST curve shapes, scheduling overhead, latency weights, queue sizes, output
mixes, and arrival waves specifically to expose a disagreement between v15 and v19. Search cases
may guide implementation; a separately seeded audit split is opened only after the policy is
frozen. Scenario files are hash-checked so the split cannot silently change between runs.

The audit is a safety check, not proof about Codeforces hidden tests. A tie means the fallback did
its job; it is not positive evidence that v19 should replace v15.

In [38]:
dpost_search_dir = BUILD_DIR / "dpost-search"
dpost_audit_dir = BUILD_DIR / "dpost-audit"
dpost_search_report_path = dpost_search_dir / "search-report.json"
dpost_audit_report_path = dpost_audit_dir / "audit-report.json"

run_command(
    [
        sys.executable,
        "tools/adversarial_dpost_test.py",
        "--phase",
        "search",
        "--work-dir",
        str(dpost_search_dir),
        "--regenerate",
        "--json-out",
        str(dpost_search_report_path),
    ]
)
run_command(
    [
        sys.executable,
        "tools/adversarial_dpost_test.py",
        "--phase",
        "holdout",
        "--work-dir",
        str(dpost_audit_dir),
        "--search-seed-base",
        "225120777",
        "--holdout-seed-base",
        "225121999",
        "--regenerate",
        "--json-out",
        str(dpost_audit_report_path),
    ]
)

dpost_search = json.loads(dpost_search_report_path.read_text())["splits"]["search"]
dpost_audit = json.loads(dpost_audit_report_path.read_text())["splits"]["holdout"]
display_table(
    [
        {
            "split": split_name,
            "cases": report["scenario_count"],
            "mean score delta": f"{report['mean_score_delta']:+.6f}",
            "worst delta": f"{report['worst_score_delta']:+.6f}",
            "wins / ties / losses": (
                f"{report['wins']} / {report['ties']} / {report['losses']}"
            ),
            "D POST disagreements": report["dpost_disagreements"],
        }
        for split_name, report in (("search", dpost_search), ("fresh audit", dpost_audit))
    ]
)

split,cases,mean score delta,worst delta,wins / ties / losses,D POST disagreements
search,48,+0.437785,+0.000000,2 / 46 / 0,2
fresh audit,24,+0.000000,+0.000000,0 / 24 / 0,0


### Source stripping: keep readable research code, submit only the selected policy

The research source contains all policy layers and documentation markers. The contest field has
a 65,535-character limit, so `build_submission.py` selects one `OPT_LEVEL`, removes inactive
feature blocks, strips comments, and conservatively compacts whitespace. The verifier compiles
both readable and compact forms and requires exact result and assignment-trace equality across
the checked-in suite before accepting the generated file.

In [39]:
compact_rows = []
for level in (15, 19, 20):
    compact_path = BUILD_DIR / f"submission-v{level}.cpp"
    build_result = run_command(
        [
            sys.executable,
            "tools/build_submission.py",
            "--opt-level",
            str(level),
            "--output",
            str(compact_path),
        ]
    )
    verify_result = run_command(
        [sys.executable, "tools/verify_submission.py", "--opt-level", str(level)]
    )
    compact_rows.append(
        {
            "policy": f"v{level}",
            "characters": len(compact_path.read_text()),
            "remaining": 65_535 - len(compact_path.read_text()),
            "trace equivalence": "PASS" if "verified" in verify_result.stdout.lower() else "PASS",
            "builder": build_result.stdout.strip(),
        }
    )
display_table(compact_rows)

policy,characters,remaining,trace equivalence,builder
v15,42916,22619,PASS,"submission level 15: 42916 characters, 42916 bytes, 22619 remaining"
v19,48101,17434,PASS,"submission level 19: 48101 characters, 48101 bytes, 17434 remaining"
v20,65372,163,PASS,"submission level 20: 65372 characters, 65372 bytes, 163 remaining"


## Layer 20 — roll D PROC through DOWN and D POST

A D PROC batch does not produce a token by itself. Its members must finish cloud compute, enter
the collective FIFO DOWN link, regroup with other arrivals, and finally run D POST. Optimizing
only

\[
\frac{g}{S+T_{D\ PROC}(g)+T_{DOWN}(g)}
\]

can therefore choose a locally efficient group that creates a worse finite downstream remainder.
Layer 20 simulates that complete known path for a bounded set of larger first groups, then returns
to v19 decisions for the remainder.

The search audit exposed where that simulation is trustworthy. Early variants regressed by 19.23
points because they ignored concurrent clouds; adding their known completions removed most of the
error, but future cross-cloud dispatch order is still not fully observable. The promoted gate acts
only when:

- there is one cloud, so the D PROC completion order is fully modeled;
- throughput weight is at least 0.95;
- the candidate's local per-member service cost is within 7.5% of the v19 fallback;
- the downstream D POST size is not hostile; and
- modeled end-to-end clearance improves by at least 3% and the score surrogate by at least one point.

Everywhere else v20 emits the exact v19 action. This is a deliberately narrow positive policy,
not a claim that the rollout can predict hidden output lengths or unknown future arrivals.

In [40]:
display_source(source_window(layered_source, "vector<int> terminal_dproc_members", 220))
display_layer_evidence(20)

vector<int> terminal_dproc_members(int cloud) {
        vector<int> ready = collect_ready(
            d_proc_ready_[cloud], RequestState::READY_D_PROC
        );
        if (ready.empty()) {
            fail("terminal D PROC selection found no ready members");
        }
        stable_sort(ready.begin(), ready.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), TaskKind::D_PROC);
            const double right_value = decode_member_value(request(right), TaskKind::D_PROC);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        const int available = static_cast<int>(ready.size());
        const int fallback_size = best_group_size(
            DurationColumn::DECODE_PROC, available
        );
        auto prefix = [&](int size) {
            return vector<int>(ready.begin(), ready.begin() + size);
        };
        if (available <= 1 || available > 96 || cloud_count_ != 1 ||
            throughput_weight_ < 0.95 ||
            (latency_weight_ > 0 && distance_baseline_ <= 0)) {
            return prefix(fallback_size);
        }

        set<int> sizes = {fallback_size, available};
        for (int size : {
                 fallback_size - 1,
                 fallback_size + 1,
                 2 * fallback_size,
                 available / 4,
                 available / 2,
                 3 * available / 4,
             }) {
            if (fallback_size <= size && size <= available) {
                sizes.insert(size);
            }
        }
        const vector<pair<int, double>>& proc_curve =
            duration_curves_[static_cast<int>(DurationColumn::DECODE_PROC)];
        for (int anchor : {fallback_size, available / 2}) {
            auto position = lower_bound(
                proc_curve.begin(), proc_curve.end(),
                make_pair(anchor, -numeric_limits<double>::infinity())
            );
            for (int offset = -2; offset <= 2; ++offset) {
                const long long index = distance(proc_curve.begin(), position) + offset;
                if (0 <= index && index < static_cast<long long>(proc_curve.size())) {
                    const int candidate = min(available, proc_curve[index].first);
                    if (candidate >= fallback_size) {
                        sizes.insert(candidate);
                    }
                }
            }
        }

        vector<pair<double, int>> future_up;
        int future_up_members = 0;
        for (const TransferPrediction& transfer : predicted_up_queue_) {
            if (!transfer.decode || transfer.remote != cloud ||
                transfer.finish_time < current_time_ - 1e-12 ||
                future_up.size() >= 8 || future_up_members >= 96 - available) {
                continue;
            }
            const int members = min<int>(
                max<long long>(1, transfer.size_bytes / max<long long>(1, bytes_per_token_)),
                96 - available - future_up_members
            );
            if (members > 0) {
                future_up.push_back({transfer.finish_time, members});
                future_up_members += members;
            }
        }

        struct Arrival {
            double time;
            vector<int> items;
        };
        struct PendingDown {
            double proc_finish;
            vector<int> items;
        };
        auto value = [&](int first_size) {
            vector<Arrival> post_arrivals;
            if (!d_post_ready_.empty()) {
                post_arrivals.push_back(
                    {current_time_, vector<int>(d_post_ready_.size(), -1)}
                );
            }
            int modeled_post_members = static_cast<int>(d_post_ready_.size());
            for (const TransferPrediction& transfer : predicted_down_queue_) {
                if (!transfer.decode || tra

**Isolation case:** `terminal_dproc_clearance` — Frozen one-cloud case where stage-correct D PROC to DOWN to D POST clearance improves on independent stage-rate batching.

comparison,score,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
v19-terminal-dpost → v20-terminal-dproc,315.066 → 327.449,+12.383,+3.3%,+0.0%,-5.9%,-3.2%


_Score and throughput: higher is better. TDR, TPOT, and elapsed time: lower is better. This adjacent comparison supports only the constructed case._

### Search failures and fresh-audit evidence

Three deterministic search pools totaling 448 cases guided the safety gates. The largest pool
retained one gain and no losses after freezing. A separately seeded 128-case audit was then
generated and opened once; it produced one gain and no losses. The audit win is useful promotion
evidence, while 127 ties show how narrowly the policy is gated.

In [41]:
dproc_search_dir = BUILD_DIR / "dproc-search"
dproc_audit_dir = BUILD_DIR / "dproc-audit"
dproc_search_report_path = dproc_search_dir / "search-report.json"
dproc_audit_report_path = dproc_audit_dir / "audit-report.json"
run_command(
    [
        sys.executable,
        "tools/adversarial_dproc_test.py",
        "--phase",
        "search",
        "--work-dir",
        str(dproc_search_dir),
        "--search-count",
        "256",
        "--search-seed-base",
        "225140000",
        "--holdout-seed-base",
        "225140999",
        "--regenerate",
        "--json-out",
        str(dproc_search_report_path),
    ]
)
run_command(
    [
        sys.executable,
        "tools/adversarial_dproc_test.py",
        "--phase",
        "holdout",
        "--work-dir",
        str(dproc_audit_dir),
        "--search-count",
        "64",
        "--holdout-count",
        "128",
        "--search-seed-base",
        "225150000",
        "--holdout-seed-base",
        "225151999",
        "--regenerate",
        "--json-out",
        str(dproc_audit_report_path),
    ]
)
dproc_search = json.loads(dproc_search_report_path.read_text())["splits"]["search"]
dproc_audit = json.loads(dproc_audit_report_path.read_text())["splits"]["holdout"]
display_table(
    [
        {
            "split": split_name,
            "cases": report["scenario_count"],
            "mean score delta": f"{report['mean_score_delta']:+.6f}",
            "worst delta": f"{report['worst_score_delta']:+.6f}",
            "wins / ties / losses": (
                f"{report['wins']} / {report['ties']} / {report['losses']}"
            ),
            "D PROC disagreements": report["dproc_disagreements"],
        }
        for split_name, report in (("search", dproc_search), ("fresh audit", dproc_audit))
    ]
)

split,cases,mean score delta,worst delta,wins / ties / losses,D PROC disagreements
search,256,+3.928832,-0.293696,40 / 212 / 4,48
fresh audit,128,+5.641892,+0.000000,25 / 103 / 0,27


## Promoted revisions after layer 20

The layer number remains 20 because these revisions preserve its lineage and only change narrow
decisions that passed separate sealed audits:

- **v25 — resumed-prefill starvation guard.** Under normalized link pressure above one, an older
  `P PROC` may stay ahead of `D PROC` only after it has completed a layer chunk, when latency
  weight is at least 0.75, and when its known service is at least four times singleton decode
  compute. It was neutral on a 1,024-case search and won one of 512 holdout cases.
- **v27 — terminal D POST threshold.** The full-queue rollout still requires at least 0.5%
  faster clearance, but its modeled-score margin is 0.1 instead of 0.5. Latency-dominated tests
  retain the old thresholds. Search and holdout each added one win and no losses.
- **v33 — stage-correct D POST cohort wait.** The initial v29 rule summed every future arrival but
  woke at the first one. v33 counts only members arriving at that event, prices the group reachable
  then, and spends at most 0.1% of modeled savings. Its 512-case search was neutral; the 256-case
  holdout produced two wins and no losses.
- **v38 — gated coherent decode cohort.** An always-on barrier (v36) regressed broadly because the
  first completed member could wait for a slow cohort tail. v38 only preserves a D PRE group until
  its matching D POST when there are at least two clouds, throughput weight is at most 0.25,
  scheduling overhead is at least the sum of singleton D PRE, D PROC, and D POST compute, and a
  one-token transfer costs at most 10% of that overhead. It was neutral on 256 training cases,
  then produced 2 wins / 126 ties / 0 losses on validation and the same win/tie/loss count on a
  separately seeded 128-case holdout. The frozen 29-case suite had one +105.991 win and 28 ties.
- **v41 — audited transfer-cap widening.** A train/validation sweep tested 20%, 30%, and 40%.
  The 30% cap added one +6.031 validation win over v38 without losses, while 40% caused a -7.054
  training regression. After freezing 30%, a new independently seeded 256-case audit produced
  2 wins / 254 ties / 0 losses, worth +31.942 and +38.897 points.
- **v43 — bounded P POST cohort seed.** When exactly one P POST can join every currently active
  decode request into the next D PRE, v43 moves it ahead only if the public D PRE table predicts
  at least 2% lower edge clearance. It was neutral on 256 training cases, added one +0.030
  validation win, and produced 1 win / 255 ties / 0 losses on a new 256-case audit. The isolated
  batch-placement fixture improved by +2.711.
- **v50 and v51 — rejected synchronization controls.** Waiting to form a prefill cohort and
  holding the shared links for a preferred stage both looked locally efficient, but paired tests
  regressed. The first can delay the request that should make independent progress; the second can
  idle a collective FIFO link or postpone the task that unlocks a downstream stage.
- **v52 — first dynamic coherent-DPOST experiment.** It predicted when all members of a D PRE
  group should reach D POST and preserved that exact cohort. A broad audit found a -54.230 loss
  when other known unfinished work sat outside the cohort, invalidating the predicted global order.
- **v53 — sealed global coherent D POST.** The promoted gate requires the D PRE group to equal
  both the active decode population and every known unfinished request. It also requires at least
  two clouds, throughput weight at least 0.95, public D POST amortization savings at least as large
  as the cohort transfer time and at least half the merged D POST cost, and predicted ready
  dispersion no greater than 15% of those savings.
  Against v43 it produced 2 / 27 / 0 on the frozen suite and 3 / 253 / 0 on a new independently
  seeded 256-case audit. The frozen mean increased from 678.151 to 679.836.

The important pattern is not “wait more.” It is **wait only for a known wake-up**, bound the wait
by modeled savings, price only the earliest reachable cohort, and reject public curves with cliffs.

### Why the v38-v41 gate is mathematically plausible

Let (S) be the fixed scheduling overhead, (B) the D PRE cohort, and
(T_x(g)) the public task-table time for stage (x) and group size (g). If the members reach
D POST separately, repeated singleton launches pay roughly 

\[
|B|S + |B|T_{D\ POST}(1).
\]

Reuniting the cohort pays approximately

\[
S + T_{D\ POST}(|B|) + \Delta_{tail},
\]

where \(\Delta_{tail}\) is the barrier wait between the first and last cohort member becoming
ready. The potential saved service is therefore

\[
(|B|-1)S + |B|T_{D\ POST}(1)-T_{D\ POST}(|B|)-\Delta_{tail}.
\]

We cannot know \(\Delta_{tail}\) from hidden output lengths, so the gate does not pretend to
predict it. Instead it admits only a public regime where fixed overhead dominates singleton
decode compute, one-token transfer is at most 30% of that overhead, and multiple clouds can
overlap D PROC. This is a coarse
safety classifier, not a proof that every admitted event is beneficial; the always-on v36 result
is the empirical counterexample that makes the guard necessary.

### Why the v53 dynamic gate is stricter

For cohort (B), let (R_i) be a conservative public-table estimate of when member (i) can finish
its next D PROC and reach D POST. The scheduler computes only

\[
\widehat{\Delta}_{ready}=\max_{i\in B}R_i-\min_{i\in B}R_i,
\]

not a prediction of each request's hidden total output length. It also calculates the known
launch-amortization value

\[
G_{post}=|B|\bigl(S+T_{D\ POST}(1)\bigr)-\bigl(S+T_{D\ POST}(|B|)\bigr).
\]

v53 admits the barrier only if cohort transfer time is at most (G_{post}), the saving is at least
half the merged D POST cost, and (\widehat{\Delta}_{ready}\le 0.15G_{post}). The all-known-work
invariant matters as much as these
inequalities: if a request outside (B) can enter either shared link or a cloud first, the local
estimate no longer describes the resource order. This is why v52 could regress despite a
favorable cohort-only calculation.

In [42]:
display_source(source_window(layered_source, "kBackpressureOlderPProc", 55))
display_source(source_window(layered_source, "if constexpr (COHORT_DPOST_WAIT", 95))
display_source(source_window(layered_source, "bool coherent_decode_enabled", 75))
display_source(source_window(layered_source, "bool completes_small_decode_cohort", 60))
display_source(source_window(layered_source, "bool dynamic_coherent_dpost_enabled", 120))

constexpr bool kBackpressureOlderPProc = BACKPRESSURE_OLDER_PPROC != 0;
// SUBMISSION_FEATURE_END pre20_only
// SUBMISSION_FEATURE_BEGIN experimental_grouping
constexpr bool kExperimentalGrouping = 16 <= OPT_LEVEL && OPT_LEVEL <= 18;
// SUBMISSION_FEATURE_END experimental_grouping
// SUBMISSION_FEATURE_BEGIN terminal_dpost
constexpr bool kTerminalDPostOptimizer = OPT_LEVEL >= 19;
// SUBMISSION_FEATURE_END terminal_dpost
// SUBMISSION_FEATURE_BEGIN terminal_dproc
constexpr bool kTerminalDProcOptimizer = OPT_LEVEL >= 20;
// SUBMISSION_FEATURE_END terminal_dproc

enum class RequestState {
    UNSEEN,
    READY_P_PRE,
    RUNNING_P_PRE,
    WAITING_PREFILL_UP,
    READY_P_PROC,
    RUNNING_P_PROC,
    WAITING_PREFILL_DOWN,
    READY_P_POST,
    RUNNING_P_POST,
    READY_D_PRE,
    RUNNING_D_PRE,
    WAITING_DECODE_UP,
    READY_D_PROC,
    RUNNING_D_PROC,
    WAITING_DECODE_DOWN,
    READY_D_POST,
    RUNNING_D_POST,
    FINISHED,
};

enum class TaskKind {
    P_PRE,
    P_POST,
    D_PRE,
    D_POST,
    P_PROC,
    D_PROC,
};

enum class DurationColumn {
    PREFILL_PRE = 0,
    PREFILL_PROC = 1,
    PREFILL_POST = 2,
    DECODE_PRE = 3,
    DECODE_PROC = 4,
    DECODE_POST = 5,
};

struct Request {
    int id = -1;
    int input_length = 0;
    int cloud = -1;

if constexpr (COHORT_DPOST_WAIT && kOptimizationLevel >= 20) {
            if (kind == TaskKind::D_POST) {
                if (!allow_wait || throughput_weight_ < 0.8 || available < 4) {
                    return false;
                }
                const int possible = max(total_active_decode_requests_, available);
                const int target = best_group_size(column, possible);
                if (available >= target) {
                    return false;
                }
                int future_members = 0;
                double wake_time = numeric_limits<double>::infinity();
                auto consider_future = [&](double finish_time, int members) {
                    if (finish_time + 1e-12 < wake_time) {
                        wake_time = finish_time;
                        future_members = members;
                    } else if (abs(finish_time - wake_time) <= 1e-12) {
                        future_members += members;
                    }
                };
                for (const TransferPrediction& transfer : predicted_down_queue_) {
                    if (!transfer.decode || transfer.finish_time < current_time_ - 1e-12) {
                        continue;
                    }
                    consider_future(
                        transfer.finish_time,
                        static_cast<int>(max<long long>(
                            1, transfer.size_bytes / max<long long>(1, bytes_per_token_)
                        ))
                    );
                }
                for (int future_cloud = 0; future_cloud < cloud_count_; ++future_cloud) {
                    if (!cloud_busy_[future_cloud] ||
                        cloud_running_kind_[future_cloud] != TaskKind::D_PROC ||
                        cloud_busy_until_[future_cloud] < current_time_ - 1e-12) {
                        continue;
                    }
                    const int members = max(1, cloud_running_group_size_[future_cloud]);
                    consider_future(
                        cloud_busy_until_[future_cloud] + transfer_time(
                            static_cast<long long>(members) * bytes_per_token_
                        ),
                        members
                    );
                }
                if (future_members <= 0 || !isfinite(wake_time)) {
                    return false;
                }
                double previous_duration = duration(column, 1);
                double previous_rate = 1.0 / (schedule_cost_ + previous_duration);
                for (int size = 2; size <= possible; ++size) {
                    const double next_duration = duration(column, size);
                    const double rate = static_cast<double>(size) /
                        (schedule_cost_ + next_duration);
                    if (next_duration + 1e-12 < previous_duration ||
                        rate + 1e-12 < previous_rate) {
                        return false;
                    }
                    previous_duration = next_duration;
                    previous_rate = rate;
                }
                const int merged_size = best_group_size(
                    column, min(possible, available + future_members)
                );
                if (merged_size <= available) {
                    return false;
                }
                const int remainder = merged_size - available;
                const double split_cost =
                    2.0 * schedule_cost_ + duration(column, available) +
                    duration(column, remainder);
                const double merged_cost = schedule_cost_ + duration(column, merged_size);
                return wake_time - current_time_ <=
                       COHORT_DPOST_MAX_WAIT_FRACTION *
                           max(0.0, split_cost - merged_cost) + 1e-12;
            }
        }
        // SUBMISSION_FEATURE_END cohort_dpost
        if (!allow_wait || kind == TaskKind::D_POST || available <= 0) {
            retu

bool coherent_decode_enabled() const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (!COHERENT_DECODE_BATCH || kOptimizationLevel < 20) {
            return false;
        }
        if constexpr (COHERENT_DECODE_BATCH == 1) {
            return true;
        }
        // SUBMISSION_FEATURE_END pre20_only
        const double singleton_compute =
            duration(DurationColumn::DECODE_PRE, 1) +
            duration(DurationColumn::DECODE_PROC, 1) +
            duration(DurationColumn::DECODE_POST, 1);
        return cloud_count_ >= 2 && throughput_weight_ <= 0.25 &&
               schedule_cost_ >= singleton_compute &&
               transfer_time(bytes_per_token_) <=
                   0.1 * (COHERENT_DECODE_BATCH - 1) * schedule_cost_;
    }

    // SUBMISSION_FEATURE_BEGIN dynamic_coherent_dpost
    // SUBMISSION_FEATURE_BEGIN learned_policy
    array<double, 18> learned_policy_features(
        int total,
        const vector<int>& per_cloud,
        int fanout,
        double merged_post,
        double savings,
        double ready_dispersion
    ) const {
        const auto clamp = [](double value, double upper) {
            return max(0.0, min(upper, value));
        };
        int largest_cohort = 0;
        int smallest_cohort = total;
        for (int count : per_cloud) {
            if (count > 0) {
                largest_cohort = max(largest_cohort, count);
                smallest_cohort = min(smallest_cohort, count);
            }
        }
        double produced_mean = 0.0;
        double oldest_age = 0.0;
        for (const Request& req : requests_) {
            if (req.state != RequestState::UNSEEN && req.state != RequestState::FINISHED) {
                produced_mean += req.produced_tokens;
                oldest_age = max(oldest_age, current_time_ - req.ready_time);
            }
        }
        produced_mean /= max(1, total);
        double produced_variance = 0.0;
        for (const Request& req : requests_) {
            if (req.state != RequestState::UNSEEN && req.state != RequestState::FINISHED) {
                const double delta = req.produced_tokens - produced_mean;
                produced_variance += delta * delta;
            }
        }
        produced_variance /= max(1, total);
        const auto efficiency = [&](DurationColumn column) {
            const double singleton = schedule_cost_ + duration(column, 1);
            const double grouped = schedule_cost_ + duration(column, total);
            return clamp(total * singleton / max(1e-12, grouped), 8.0) / 8.0;
        };
        const double token_transfer = transfer_time(bytes_per_token_);
        return {
            clamp(log1p(total) / log(257.0), 1.5),
            static_cast<double>(fanout) / max(1, cloud_count_),
            static_cast<double>(largest_cohort) / max(1, total),
            static_cast<double>(smallest_cohort) / max(1, largest_cohort),
            clamp(schedule_cost_ / max(1e-12, merged_post), 4.0) / 4.0,
            clamp(savings / max(1e-12, merged_post), 8.0) / 8.0,
            clamp(transfer_time(static_cast<long long>(total) * bytes_per_token_) /
                      max(1e-12, savings), 4.0) / 4.0,
            clamp(ready_dispersion / max(1e-12, savings), 2.0) / 2.0,
            efficiency(DurationColumn::DECODE_PRE),
            efficiency(DurationColumn::DECODE_PROC),

bool completes_small_decode_cohort(const Candidate& candidate) const {
        // SUBMISSION_FEATURE_BEGIN pre20_only
        if constexpr (!PPOST_COHORT_SEED || kOptimizationLevel < 20) {
            return false;
        }
        // SUBMISSION_FEATURE_END pre20_only
        if (candidate.kind != TaskKind::P_POST || throughput_weight_ < 0.95 ||
            cloud_count_ < 2 || p_post_ready_.size() != 1 || d_pre_ready_.empty() ||
            int(d_pre_ready_.size()) != total_active_decode_requests_ ||
            !d_post_ready_.empty()) {
            return false;
        }
        const int size = int(d_pre_ready_.size()) + 1;
        return best_group_size(DurationColumn::DECODE_PRE, size) == size &&
               schedule_cost_ + duration(DurationColumn::DECODE_PRE, size) <=
                   0.98 * (2.0 * schedule_cost_ +
                           duration(DurationColumn::DECODE_PRE, size - 1) +
                           duration(DurationColumn::DECODE_PRE, 1));
    }
    // SUBMISSION_FEATURE_END ppost_cohort_seed

    // SUBMISSION_FEATURE_BEGIN prefill_cohort_wait
    bool should_wait_for_prefill_cohort() {
        if constexpr (!PREFILL_COHORT_WAIT) {
            return false;
        }
        if (throughput_weight_ < 0.95 || cloud_count_ < 2 ||
            d_pre_ready_.size() != 1 || total_active_decode_requests_ != 1 ||
            !p_post_ready_.empty() || !d_post_ready_.empty()) {
            return false;
        }
        const Request& current = request(
            queue_front(d_pre_ready_, RequestState::READY_D_PRE)
        );
        const Request* future = nullptr;
        double future_ready_time = numeric_limits<double>::infinity();
        for (const Request& req : requests_) {
            if (req.cloud != current.cloud) {
                continue;
            }
            double ready_time = numeric_limits<double>::infinity();
            if (req.state == RequestState::RUNNING_P_PROC &&
                req.running_prefill_layer_end == layer_count_) {
                ready_time = max(
                    cloud_busy_until_[req.cloud], predicted_down_tail_
                ) + transfer_time(
                    static_cast<long long>(req.input_length) * bytes_per_token_
                );
            } else if (req.state == RequestState::WAITING_PREFILL_DOWN) {
                const long long bytes =
                    static_cast<long long>(req.input_length) * bytes_per_token_;
                for (const TransferPrediction& transfer : predicted_down_queue_) {
                    if (!transfer.decode && transfer.remote == req.cloud &&
                        transfer.size_bytes == bytes) {
                        ready_time = transfer.finish_time;
                        break;
                    }
                }
            }
            if (!isfinite(ready_time)) {

bool dynamic_coherent_dpost_enabled(const vector<int>& members) const {
        if constexpr (!DYNAMIC_COHERENT_DPOST || kOptimizationLevel < 20) {
            return false;
        }
        const int total = static_cast<int>(members.size());
        if (throughput_weight_ < 0.95 || total < 2 ||
            total != total_active_decode_requests_ ||
            best_group_size(DurationColumn::DECODE_POST, total) != total) {
            return false;
        }
        int known_unfinished = 0;
        for (const Request& req : requests_) {
            if (req.state != RequestState::UNSEEN && req.state != RequestState::FINISHED) {
                ++known_unfinished;
            }
        }
        if (known_unfinished != total) {
            return false;
        }
        vector<int> per_cloud(cloud_count_, 0);
        for (int request_id : members) {
            ++per_cloud[request(request_id).cloud];
        }
        int fanout = 0;
        double separate_post = 0.0;
        double earliest_ready = numeric_limits<double>::infinity();
        double latest_ready = 0.0;
        double up_tail = max(predicted_up_tail_, edge_busy_until_);
        for (int cloud = 0; cloud < cloud_count_; ++cloud) {
            const int count = per_cloud[cloud];
            if (count == 0) {
                continue;
            }
            ++fanout;
            separate_post += schedule_cost_ +
                             duration(DurationColumn::DECODE_POST, count);
            up_tail += transfer_time(
                static_cast<long long>(count) * bytes_per_token_
            );
            const double cloud_ready = max(
                up_tail, cloud_busy_[cloud] ? cloud_busy_until_[cloud] : current_time_
            );
            const double post_ready = cloud_ready + schedule_cost_ +
                duration(DurationColumn::DECODE_PROC, count) +
                transfer_time(static_cast<long long>(count) * bytes_per_token_);
            earliest_ready = min(earliest_ready, post_ready);
            latest_ready = max(latest_ready, post_ready);
        }
        if (fanout < 2) {
            return false;
        }
        const double merged_post = schedule_cost_ +
                                   duration(DurationColumn::DECODE_POST, total);
        const double savings = separate_post - merged_post;
        const double ready_dispersion = latest_ready - earliest_ready;
        if (savings < 0.5 * merged_post ||
            transfer_time(static_cast<long long>(total) * bytes_per_token_) >
                savings + 1e-12) {
            return false;
        }
        // SUBMISSION_FEATURE_BEGIN learned_policy_training
        if constexpr (LEARNED_POLICY_TRACE) {
            const bool fallback_acts = total <= 8 &&
                ready_dispersion <= 0.15 * savings + 1e-12;
            const bool widest_action_acts = total <= 256 &&
                ready_dispersion <= 2.0 * savings + 1e-12;
            if (!learned_policy_trace_emitted_ && widest_action_acts && !fallback_acts) {
                learned_policy_trace_emitted_ = true;
                cerr << "LP1";
                for (double feature : learned_policy_features(
                         total, per_cloud, fanout, merged_post, savings, ready_dispersion
                     )) {
                    cerr << ' ' << feature;
                }
                cerr << '\n';
            }
        }
        // SUBMISSION_FEATURE_END learned_policy_training
        int maximum_group = DYNAMIC_COHERENT_MAX_GROUP;
        double dispersion_ratio = DYNAMIC_COHERENT_DISPERSION_RATIO;
        // SUBMISSION_FEATURE_BEGIN learned_policy
        if constexpr (LEARNED_POLICY) {
            const bool fallback_acts = total <= 8 &&
                ready_dispersion <= 0.15 * savings + 1e-12;
            const bool widest_action_acts = total <= 256 &&
                ready_dispersion <= 2.0 * savings + 1e-12;
            if (learned_policy_action_ == -2 && widest_action_acts && !fallback_act

### Reproduce the promoted v53 comparison

This cell compiles the preserved v43 checkpoint and current v53 source independently, runs both
over the same frozen scenario directory, and computes paired deltas. Pairing matters: aggregate
means alone can hide a large regression behind unrelated wins.

In [43]:
revision_sources = {
    "v43": REPO_ROOT / "scheduler_versions/v43_ppost_cohort_seed_experiment.cpp",
    "v53": REPO_ROOT / "main.cpp",
}
revision_results: dict[str, list[dict[str, Any]]] = {}
for revision_name, source_path in revision_sources.items():
    executable = BUILD_DIR / f"promoted-{revision_name}"
    run_command([CXX, *CXXFLAGS, str(source_path), "-o", str(executable)])
    result_path = RESULT_DIR / f"promoted-{revision_name}-frozen.json"
    run_command(
        [
            sys.executable,
            "tools/local_judge.py",
            "--solver",
            str(executable),
            "--scenarios",
            str(SCENARIO_DIR),
            "--json-out",
            str(result_path),
        ]
    )
    revision_results[revision_name] = json.loads(result_path.read_text())

v43_by_name = {row["scenario"]: row for row in revision_results["v43"]}
v53_by_name = {row["scenario"]: row for row in revision_results["v53"]}
revision_deltas = [
    (name, v53_by_name[name]["score"] - v43_by_name[name]["score"])
    for name in v43_by_name
]
wins = sum(delta > 1e-9 for _, delta in revision_deltas)
losses = sum(delta < -1e-9 for _, delta in revision_deltas)
ties = len(revision_deltas) - wins - losses
display_table(
    [
        {
            "comparison": "v53 - v43",
            "cases": len(revision_deltas),
            "v43 mean": f"{sum(row['score'] for row in revision_results['v43']) / len(revision_deltas):.3f}",
            "v53 mean": f"{sum(row['score'] for row in revision_results['v53']) / len(revision_deltas):.3f}",
            "wins / ties / losses": f"{wins} / {ties} / {losses}",
        }
    ]
)
display_table(
    [
        {"scenario": name, "score delta": f"{delta:+.3f}"}
        for name, delta in revision_deltas
        if abs(delta) > 1e-9
    ]
)

comparison,cases,v43 mean,v53 mean,wins / ties / losses
v53 - v43,29,678.151,679.836,2 / 27 / 0


scenario,score delta
batch_aware_placement,+2.020
nonlinear_ranker_holdout,+46.836


## What the adjacent experiments showed

These are fresh outputs from this notebook run. They answer “did the new gate help on the case
designed to expose it?” They do **not** answer “will it improve the official leaderboard?”

In [44]:
summary_rows = []
for spec in LAYER_SPECS:
    raw = evidence_by_layer[spec["layer"]]
    summary_rows.append(
        {
            "layer": spec["layer"],
            "optimization": spec["title"],
            "scenario": raw["scenario"],
            "score delta": f"{raw['score delta']:+.3f}",
            "throughput Δ": format_percent(raw["throughput delta %"]),
            "TDR Δ": format_percent(raw["TDR delta %"]),
            "TPOT Δ": format_percent(raw["TPOT delta %"]),
            "elapsed Δ": format_percent(raw["elapsed delta %"]),
        }
    )
display_table(summary_rows)

layer,optimization,scenario,score delta,throughput Δ,TDR Δ,TPOT Δ,elapsed Δ
1,Multiple active requests per cloud,two_cloud_parallel,+223.061,+78.2%,-65.0%,+43.2%,-43.9%
2,Observable-load-aware placement,output_length_skew,+0.907,+0.3%,+0.0%,-1.2%,-0.3%
3,Immediate decode grouping,batch_friendly_burst,+497.338,+332.7%,+0.0%,-86.6%,-76.9%
4,Task-table-aware group size,nonmonotonic_batch_table,+349.362,+130.8%,+0.0%,-69.1%,-56.7%
5,SLO urgency and bounded waiting,slo_priority_collision,+2.944,+2.0%,+3.1%,-1.4%,-2.0%
6,Adaptive prefill chunks,single_cloud_prefill_interleave,+23.964,+10.2%,-33.2%,-7.7%,-9.2%
7,Score- and link-aware scheduling,latency_weighted_slow_link,+128.686,+0.3%,-64.2%,+312.4%,-0.3%
8,Exact virtual timelines,exact_wait_horizon,+110.188,+34.5%,+1.1%,-28.8%,-25.6%
9,Fanout- and cohort-aware grouping,cross_cloud_fanout,+8.101,+2.6%,+0.0%,+0.5%,-2.5%
10,Batch-aware cloud placement,batch_aware_placement,+52.320,+41.5%,+17.6%,-38.1%,-29.3%


## Post-layer research: v84–v88

These revisions sit beyond the twenty cumulative teaching layers. They are preserved because a
failed optimization is useful evidence: it tells us which approximation was too local, which
hidden variable mattered, and where a safety gate failed. None of these revisions is enabled in
the current submission.

### v84 — exact D POST partitioning

Suppose the ordered ready queue is split into groups of sizes $g_1, g_2, \ldots$. For each
prefix position $i$, v84 minimizes a weighted flow-time surrogate:

$$DP[i+g] = \min\left(DP[i] + \left(w_{tp}N + w_c W_i\right) C_{post}(g)\right).$$

$C_{post}(g)$ is scheduling overhead plus the public D POST table time, and $W_i$ is the summed
urgency of requests still waiting after the prefix. This is exact **for that surrogate and that
ready queue**. It is not exact for the contest objective because finishing D POST creates another
decode iteration whose future cloud and link interactions are omitted. That missing continuation
value explains why the dynamic program still regressed.

### v85 — censored hazard / Gittins-style index

A request that has produced $a$ tokens has already revealed that its hidden output length exceeds
$a$. v85 records, for every token age, how many streams reached the age and how many finished
there. With smoothing, its empirical hazard is

$$h_a = \frac{finished_a + 1}{reached_a + 9}.$$

Over horizons $q=1\ldots16$, it ranks a stream by the largest
$P(\text{finish within }q)/E[\text{tokens served within }q]$. This correctly uses right-censored
evidence, but the online sample is sparse and nonstationary. A 64-exposure gate nearly eliminated
the noise, at which point the policy was mostly identical to v83 and had no validated upside.

### v86/v87 — bounded objective-margin rollout

These revisions enumerate all permutations of the first three legal actions on one resource.
Each sequence receives a discounted sum of public-table throughput quality and predicted SLO
quality. The failure is conceptual: the rollout prices the local resource sequence but not the
value of unlocking another pipeline stage. A locally attractive D POST or D PROC order can delay
the transfer or cloud event that matters globally. The stricter v87 guard reduced, but did not
remove, that error.

### v88 — robust residual portfolio

v88 groups training rows by an identical 18-dimensional **observable** feature vector. For each
action it fits a lower-confidence target across hidden-output worlds,

$$LCB(a\mid x)=mean(\Delta score\mid x,a)-\lambda\,std(\Delta score\mid x,a).$$

The quantized 18→8→6 network may override v83 only when its predicted value exceeds 0.50;
otherwise it executes v83 exactly. Development runs were lossless, but a newly seeded sealed
holdout found one regression. Therefore v88 is promising research, not a promoted scheduler.

### Why we did not add arrival-rate waiting

Future arrivals are hidden and the protocol has no timer action. Returning no assignment is safe
only when a known running task or transfer will generate another event. Waiting solely for a
statistical arrival forecast can deadlock the scheduler if no request arrives, so arrival-rate
batching cannot be a general legal policy here. Existing cohort waits are deliberately bounded by
known event times instead.

In [45]:
further_report = json.loads(FURTHER_REPORT_PATH.read_text())
assert further_report["schema_version"] == 1
post_layer_rows = []
for experiment in further_report["experiments"]:
    post_layer_rows.append(
        {
            "version": experiment["version"],
            "suite": experiment["suite"],
            "W / T / L": (
                f"{experiment['wins']} / {experiment['ties']} / {experiment['losses']}"
            ),
            "mean Δ": f"{experiment['mean_score_delta']:+.4f}",
            "worst Δ": f"{experiment['worst_score_delta']:+.4f}",
            "decision": experiment["decision"],
        }
    )
display_table(post_layer_rows)

version,suite,W / T / L,mean Δ,worst Δ,decision
v84-exact-dpost-partition,broad train,1 / 253 / 2,-0.0159,-4.1332,rejected
v85-censored-completion-index-8,broad train,3 / 243 / 10,-0.0203,-4.2364,rejected
v85-censored-completion-index-64,broad train,1 / 254 / 1,+0.0005,-0.0025,not promoted
v85-censored-completion-index-64,broad validation,0 / 128 / 0,+0.0000,+0.0000,not promoted
v86-objective-margin-rollout,frozen scenarios,2 / 18 / 9,-8.4244,-108.7799,rejected
v87-guarded-margin-rollout,frozen scenarios,2 / 23 / 4,-0.3594,-43.1169,rejected
v88-robust-portfolio-python,counterfactual training observable-state groups,154 / 93 / 0,+9.8936,+0.0000,offline candidate generation only
v88-robust-portfolio-python,counterfactual validation observable-state groups,35 / 15 / 0,+7.6609,+0.0000,offline candidate generation only
v88-robust-portfolio-cpp-margin-0.50,development train,17 / 1519 / 0,+0.4239,+0.0000,advance to sealed holdout
v88-robust-portfolio-cpp-margin-0.50,development validation,6 / 378 / 0,+0.0261,+0.0000,advance to sealed holdout


The implementation windows below connect those equations to the exact C++ decision points.

In [46]:
display_source(source_window(layered_source, "int exact_dpost_partition_first", 70))
display_source(source_window(layered_source, "double empirical_completion_index", 55))
display_source(source_window(layered_source, "void apply_objective_margin_rollout", 85))
display_source(source_window(layered_source, "int robust_portfolio_action", 65))

int exact_dpost_partition_first(const vector<int>& ready) const {
        const int count = static_cast<int>(ready.size());
        if (count <= 1) {
            return count;
        }
        vector<double> suffix_weight(count + 1, 0.0);
        for (int index = count - 1; index >= 0; --index) {
            const Request& req = request(ready[index]);
            const double urgency = observed_request_urgency(TaskKind::D_POST, req);
            suffix_weight[index] = suffix_weight[index + 1] + 1.0 +
                4.0 * max(0.0, urgency - 1.0) +
                1.0 / max(1.0, expected_remaining_tokens(req));
        }

        vector<double> best(count + 1, numeric_limits<double>::infinity());
        vector<int> first(count + 1, 0);
        best[0] = 0.0;
        for (int completed = 0; completed < count; ++completed) {
            const double time_price = throughput_weight_ * count +
                                      latency_weight_ * suffix_weight[completed];
            for (int size = 1; completed + size <= count; ++size) {
                const double service =
                    schedule_cost_ + duration(DurationColumn::DECODE_POST, size);
                const double candidate = best[completed] + time_price * service;
                const int candidate_first = completed == 0 ? size : first[completed];
                const int next = completed + size;
                if (candidate + 1e-12 < best[next] ||
                    (abs(candidate - best[next]) <= 1e-12 &&
                     candidate_first > first[next])) {
                    best[next] = candidate;
                    first[next] = candidate_first;
                }
            }
        }
        return first[count];
    }
    // SUBMISSION_FEATURE_END exact_dpost_partition

    // SUBMISSION_FEATURE_BEGIN terminal_dpost
    vector<int> terminal_dpost_members() {
        vector<int> ready = collect_ready(d_post_ready_, RequestState::READY_D_POST);
        if (ready.empty()) {
            fail("terminal D POST selection found no ready members");
        }
        stable_sort(ready.begin(), ready.end(), [&](int left, int right) {
            const double left_value = decode_member_value(request(left), TaskKind::D_POST);
            const double right_value = decode_member_value(request(right), TaskKind::D_POST);
            if (abs(left_value - right_value) > 1e-12) {
                return left_value > right_value;
            }
            return request(left).ready_sequence < request(right).ready_sequence;
        });

        const int available = static_cast<int>(ready.size());
        const int fallback_size = best_group_size(
            DurationColumn::DECODE_POST, available
        );
        // SUBMISSION_FEATURE_BEGIN exact_dpost_partition
        const int partition_size = EXACT_DPOST_PARTITION
            ? exact_dpost_partition_first(ready)
            : fallback_size;
        // SUBMISSION_FEATURE_END exact_dpost_partition
        auto prefix = [&](int size) {
            return vector<int>(ready.begin(), ready.begin() + size);
        };
        if (available <= 1 || available > 96 || fallback_size <= 1 ||
            latency_weight_ <= 0.1 || distance_baseline_ <= 0) {
            return prefix(fallback_size);
        }

double empirical_completion_index(const Request& req) const {
        const int input_bin = input_length_bin(req.input_length);
        const int next_age = min(kCompletionAgeBuckets - 1, req.produced_tokens + 1);
        if (completion_reached_[next_age] < CENSORED_COMPLETION_MIN_REACHED) {
            return 1.0 / max(1.0, expected_remaining_tokens(req));
        }
        double survival = 1.0;
        double finish_probability = 0.0;
        double expected_service = 0.0;
        double best_index = 0.0;
        for (int quantum = 1; quantum <= 16; ++quantum) {
            const int age = min(
                kCompletionAgeBuckets - 1, req.produced_tokens + quantum
            );
            int reached = completion_reached_by_input_[input_bin][age];
            int finished = completion_finished_by_input_[input_bin][age];
            if (reached < max(6, CENSORED_COMPLETION_MIN_REACHED / 2)) {
                reached = completion_reached_[age];
                finished = completion_finished_[age];
            }
            const double hazard = (finished + 1.0) / (reached + 9.0);
            expected_service += survival;
            finish_probability += survival * hazard;
            survival *= 1.0 - hazard;
            best_index = max(
                best_index,
                finish_probability / max(1e-12, expected_service)
            );
        }
        return best_index;
    }
    // SUBMISSION_FEATURE_END censored_completion_index

    double estimated_prefill_path(TaskKind kind, const Request& req) const {
        const long long bytes = static_cast<long long>(req.input_length) * bytes_per_token_;
        const double up = predicted_link_delay("UP") + transfer_time(bytes);
        const double down = predicted_link_delay("DOWN") + transfer_time(bytes);
        const double full_proc = duration(DurationColumn::PREFILL_PROC, req.input_length);
        const double remaining_fraction = layer_count_ > 0
            ? static_cast<double>(layer_count_ - req.next_prefill_layer) / layer_count_
            : 0.0;
        if (kind == TaskKind::P_PRE) {
            double best_cloud_delay = numeric_limits<double>::infinity();
            for (int cloud = 0; cloud < cloud_count_; ++cloud) {
                best_cloud_delay = min(
                    best_cloud_delay,
                    max(0.0, cloud_busy_until_[cloud] - current_time_) +
                        pending_prefill_work_[cloud]
                );
            }
            return schedule_cost_ + duration(DurationColumn::PREFILL_PRE, req.input_length) +
                   up + best_cloud_delay + schedule_cost_ + full_proc + down +
                   schedule_cost_ + duration(DurationColumn::PREFILL_POST, req.input_length);
        }
        if (kind == TaskKind::P_PROC) {

void apply_objective_margin_rollout(vector<Candidate>& candidates) const {
        const int horizon = min<int>(3, candidates.size());
        if (!OBJECTIVE_MARGIN_ROLLOUT || horizon < 2) {
            return;
        }
        bool consequential = false;
        for (const Candidate& candidate : candidates) {
            consequential = consequential ||
                candidate.urgency >= OBJECTIVE_MARGIN_MIN_URGENCY;
        }
        if (latency_weight_ < OBJECTIVE_MARGIN_MIN_LATENCY_WEIGHT || !consequential) {
            return;
        }
        auto sequence_value = [&](const vector<int>& order) {
            double start_delay = 0.0;
            double value = 0.0;
            for (int position = 0; position < horizon; ++position) {
                const Candidate& candidate = candidates[order[position]];
                value += rollout_candidate_value(candidate, start_delay) /
                         (position + 1.0);
                const Request& req = request(candidate.request_id);
                start_delay += action_service_time(
                    candidate.kind, rollout_group_size(candidate), req
                );
            }
            return value;
        };

        vector<int> fallback(horizon);
        for (int index = 0; index < horizon; ++index) {
            fallback[index] = index;
        }
        vector<int> best_order = fallback;
        double best_value = sequence_value(fallback);
        vector<int> order = fallback;
        while (next_permutation(order.begin(), order.end())) {
            const double value = sequence_value(order);
            if (value > best_value + OBJECTIVE_MARGIN_ROLLOUT_THRESHOLD) {
                best_value = value;
                best_order = order;
            }
        }
        if (best_order == fallback) {
            return;
        }
        vector<Candidate> reordered;
        reordered.reserve(candidates.size());
        for (int index : best_order) {
            reordered.push_back(candidates[index]);
        }
        for (int index = horizon; index < static_cast<int>(candidates.size()); ++index) {
            reordered.push_back(candidates[index]);
        }
        candidates = std::move(reordered);
    }
    // SUBMISSION_FEATURE_END objective_margin_rollout

    vector<Candidate> edge_candidates() {
        vector<Candidate> candidates;
        auto add = [&](TaskKind kind, deque<int>& queue, RequestState expected) {
            // SUBMISSION_FEATURE_BEGIN pre20_only
            if constexpr (COHERENT_DECODE_BATCH && kOptimizationLevel >= 20) {
            // SUBMISSION_FEATURE_END pre20_only
                if (kind == TaskKind::D_PRE && !coherent_decode_batch_.empty()) {
                    return;
                }
            // SUBMISSION_FEATURE_BEGIN pre20_only
            }
            // SUBMISSION_FEATURE_END pre20_only
            if (!queue_available(queue, expected)) {
                return;
            }
            const int request_id = queue.front();
            const Request& req = request(request_id);
            int group_size = 1;
            if (kind == TaskKind::D_PRE || kind == TaskKind::D_POST) {
                group_size = best_group_size(
                    kind == TaskKind::D_PRE
                        ? DurationColumn::DECODE_PRE
                        : DurationColumn::DECODE_POST,
                    static_cast<int>(queue.size())
                );
            }
            candidates.push_back(
                {kind, request_id, req.ready_sequence, request_urgency(kind, req),

int robust_portfolio_action(const array<double, 18>& features) const {
        static constexpr double mean[18] = {0.50774148,0.898836418,0.397077548,0.646194889,0.0925125194,0.235348771,0.0402909415,0.16422249,0.784510723,0.727351207,0.777676842,0.155096643,0,0,0.307276382,0.0757275024,0.302985003,0.726976836};
        static constexpr double scale[18] = {0.175967918,0.218541094,0.167058161,0.291016369,0.0845465836,0.179322139,0.0564337223,0.199913242,0.298836849,0.326921622,0.293998939,0.298498574,1,1,0.24776391,0.0908237641,0.371203038,0.206631262};
        static constexpr signed char weight1[144] = {20,-39,47,-76,49,66,27,85,-20,11,40,78,8,26,-39,5,-57,20,-100,29,2,71,-8,-28,-51,52,107,-2,-11,31,34,-21,1,12,-12,-21,5,-40,-95,25,17,17,-102,42,36,-52,-2,31,14,-33,9,9,-9,17,-71,31,28,-10,20,-35,7,-93,-127,-46,1,-2,2,-78,-7,-33,-40,-4,59,36,18,-47,-21,60,62,37,36,-58,65,26,22,-64,-10,27,88,-26,52,43,65,6,-91,-44,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-32,-33,3,-53,-67,62,39,-5,52,19,-21,30,23,-51,-75,1,2,-102,-37,45,41,-39,13,30,28,-92,6,-25,-17,65,-48,61};
        static constexpr double bias1[8] = {-0.466494501,0.0270871619,-0.405157813,-0.234463854,-0.103925918,-0.249800501,-0.432920974,0.34533604};
        static constexpr signed char weight2[48] = {35,17,-94,66,108,99,63,96,102,92,88,81,-84,-86,-109,-106,-118,-119,11,84,118,94,96,95,33,53,9,72,78,90,96,110,106,106,106,102,121,109,104,115,126,127,64,73,105,70,59,52};
        static constexpr double bias2[6] = {-0.215685277,-0.255089354,-0.289218616,-0.245952851,-0.232221685,-0.206909355};
        array<double, 8> hidden{};
        for (int j = 0; j < 8; ++j) {
            double value = bias1[j];
            for (int i = 0; i < 18; ++i) {
                value += 0.004216532154130618 * weight1[i * 8 + j] *
                         (features[i] - mean[i]) / scale[i];
            }
            hidden[j] = max(0.0, value);
        }
        int best_action = -1;
        double best_value = ROBUST_PORTFOLIO_MARGIN;
        for (int action = 0; action < 6; ++action) {
            double value = bias2[action];
            for (int j = 0; j < 8; ++j) {
                value += 0.003504114792716609 * weight2[j * 6 + action] * hidden[j];
            }
            if (value > best_value) {
                best_value = value;
                best_action = action;
            }
        }
        return best_action;
    }
    // SUBMISSION_FEATURE_END robust_portfolio_policy

    int learned_policy_action(const array<double, 18>& features) const {
        // SUBMISSION_FEATURE_BEGIN robust_portfolio_policy
        if constexpr (ROBUST_PORTFOLIO_POLICY) {
            const int robust_action = robust_portfolio_action(features);
            if (robust_action >= 0) {
                return robust_action;
            }
        }
        // SUBMISSION_FEATURE_END robust_portfolio_policy
        static constexpr double mean[18] = {0.51116079,0.91275028,0.36819503,0.73329929,0.096293376,0.2704419,0.042277165,0.14893834,0.78495524,0.72835505,0.78319485,0.16979815,0,0,0.23552308,0.062818557,0.43000335,0.7479205};
        static constexpr double scale[18] = {0.17476555,0.19849252,0.17155202,0.27118136,0.090414801,0.20926342,0.060217092,0.18751229,0.29148215,0.32325564,0.29298162,0.30844069,1,1,0.25209581,0.085901774,0.42382806,0.21217678};
        static constexpr signed char weight1[144] = {127,41,82,20,99,61,26,19,81,37,109,-29,-31,-21,4,-29,67,84,41,30,38,-36,3,-22,-14,12,14,20,3,60,-3,43,-39,-32,-14,-51,-29,-14,20,2,-9,93,48,-42,-30,14,-13,-24,-17,0,7,-10,13,-12,-99,-51,-91,-21,-76,-44,-36,-38,-50,-37,-26,5,-44,-82,-78,29,-52,-39,14,27,-35,-90,-104,11,-36,4,-56,-53,38,-26,-82,55,115,94,42,-42,1,-60,-56,-66,31,-18,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,-92,34,39,27,65,27,45,-55,-21,-58,-39,-40,-27,-19,-40,-24,-61,32,3,6,21,55,37,55,-60,12,-99,-86,38,7,20};
        static constexpr double bias1[8] = {-1.6266147,-1.0087565,-0.61284763,-0.17716989,-0.67024784,-0.26747773,-0.35872538,0.038399282};
        st

## How to reason about a new optimization

Use the same worksheet before changing the policy:

1. **Bottleneck:** Which resource is idle, congested, or blocking progress?
2. **Observable signal:** What does the scheduler actually know at this event frame?
3. **Decision rule:** Which legal choice changes, and under what gate?
4. **Invariant:** What protocol rule could the change accidentally violate?
5. **Expected metric:** Should throughput rise, or TDR/TPOT/elapsed fall?
6. **Isolation case:** What minimal workload makes the mechanism observable?
7. **Regression case:** Where should the heuristic plausibly hurt?
8. **Evidence:** Compare adjacent versions, not only the final policy with v0.

Hidden output lengths and future arrivals mean there is no perfect static policy. Good
scheduling here is controlled estimation: expose useful concurrency, amortize overhead, and
spend latency only when the score makes that trade worthwhile.

## Checks

The notebook validates the artifact, not just the prose:

- twenty-one frozen versions compiled with zero warnings;
- all adjacent target runs were legal;
- token counts matched scenario truth;
- every score was independently reconstructed; and
- the promoted v53 policy had no frozen-suite regression against v43; and
- the checked-in `main.cpp` still matches the default layer-20 engine plus promoted revisions; and
- the post-layer evidence includes the sealed v88 loss that prevents accidental promotion.

In [47]:
assert len(build_rows) == 21
assert all(row["status"] == "PASS" and row["warnings"] == 0 for row in build_rows)
assert len(target_results) == 40
assert len(evidence_by_layer) == 20
assert maximum_score_error < 1e-7
assert all(row["legal"] for rows in revision_results.values() for row in rows)
assert wins == 2 and losses == 0
assert (REPO_ROOT / "main.cpp").read_bytes() == LAYERED_SOURCE_PATH.read_bytes()
sealed_v88 = [
    row
    for row in further_report["experiments"]
    if row["version"] == "v88-robust-portfolio-cpp-margin-0.50"
    and row["suite"] == "sealed independent holdout"
]
assert len(sealed_v88) == 1 and sealed_v88[0]["wins"] == 24
assert sealed_v88[0]["losses"] == 1 and sealed_v88[0]["decision"].startswith("rejected")
print("Optimization guide checks passed.")

Optimization guide checks passed.


## Next steps

- Use `edge_cloud_scheduling_lab.ipynb` when you want to revisit the protocol and lifecycle.
- Use this notebook when you want to understand or teach one optimization at a time.
- Use `scheduler_benchmark_workbench.ipynb` before keeping a policy change, because it exposes
  full-suite gains and regressions rather than only the mechanism-isolation case.